# Nocturnal intra-urban LST — main pipeline

Reproduces the results in the accompanying IEEE GRSL letter.

**Run order:** cells top to bottom. Sections 1–9 build the data, models and
main sweep; 10–13 produce the letter's tables.

**Inputs.** Six MODIS night-LST CSVs and three 1 km node GeoJSONs, produced
by the Earth Engine scripts in `gee/`. Set `DATA_ROOT` in section 2.

**Runtime.** The main sweep (section 8) is the expensive step: 24 model fits,
roughly 3 hours on a Colab T4. Sections 10–12 read saved checkpoints and
take minutes.

**Determinism.** `set_seed()` fixes Python, NumPy and Torch seeds but does
not make GPU runs bit-reproducible. Reported numbers come from `seed=10`.


## 1 · Environment

In [ ]:
!pip install -q torch-geometric contextily

## 2 · Configuration

Set `DATA_ROOT` to the directory holding the six MODIS CSVs and three node GeoJSONs. Outside Colab, remove the `drive.mount` line and point `DATA_ROOT` at a local path.

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Point these at your own copy of the extracted data.
# DATA_ROOT holds the six MODIS CSVs and three node GeoJSONs
# produced by the GEE scripts in gee/ .
DRIVE_DATA = os.environ.get('DATA_ROOT', '/content/drive/MyDrive/UHI')
LOCAL_DATA = '/content/data_modis'
OUT_ROOT   = os.environ.get('OUT_ROOT', f'{DRIVE_DATA}/results')

DATASETS = [
    ('Bangalore', 'Blr', 'April',    'Blr_April_MODIS_60d.csv',    'Blr_Nodes_1km.geojson'),
    ('Bangalore', 'Blr', 'December', 'Blr_December_MODIS_60d.csv', 'Blr_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'April',    'Hyd_April_MODIS_60d.csv',    'Hyd_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'December', 'Hyd_December_MODIS_60d.csv', 'Hyd_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'April',    'Dlh_April_MODIS_60d.csv',    'Dlh_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'December', 'Dlh_December_MODIS_60d.csv', 'Dlh_Nodes_1km.geojson'),
]

os.makedirs(LOCAL_DATA, exist_ok=True)
print(f"Staging to {LOCAL_DATA}:")
for name in sorted({f for d in DATASETS for f in d[3:5]}):
    src, dst = f"{DRIVE_DATA}/{name}", f"{LOCAL_DATA}/{name}"
    if not os.path.exists(src):
        print(f"  [MISSING ON DRIVE] {name}"); continue
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print(f"  [cached ] {name}  ({os.path.getsize(dst)/1e6:.1f} MB)"); continue
    shutil.copy2(src, dst)
    print(f"  [staged ] {name}  ({os.path.getsize(dst)/1e6:.1f} MB)")

CONFIGS = [dict(city=c, short=s, month=m,
                csv=f'{LOCAL_DATA}/{csv}', geojson=f'{LOCAL_DATA}/{geo}')
           for c, s, m, csv, geo in DATASETS]

# ---- hyperparameters ---------------------------------------
# window_size 5, not 12: night passes arrive roughly 1-2 per day, so a
# 14-day window yields only 13-26 timesteps. A 12-step window would
# consume most of the series and leave almost no training samples.
#
# horizon 0 -- the target is LST at the LAST step of the window, not
# the next one. This is estimation, not forecasting. Forecasting night
# LST from its own history would be won by persistence and would let
# the model ignore morphology entirely, which is exactly how the ERA5
# design produced meaningless spatial skill.
HP = dict(window_size=5, horizon=0,
          train_frac=0.60, val_frac=0.15,      # test is the remainder
          k_neighbors=8, spatial_dim=32, lstm_hidden=64, dropout=0.2,
          epochs=200, seed=10)

NIGHT_ONLY = True
QC_STRICT = False        # True keeps only mandatory-QA==0 retrievals
MIN_NODE_COVERAGE = 0.60 # drop nodes valid at fewer than this fraction of kept times
MIN_TIME_COVERAGE = 0.10 # keep a timestamp if at least this fraction of nodes was retrieved

# ---- LST acquisition dates (Landsat, validation only) -------
LST_SCENES = {
    ('Delhi', 'April'):    '2025-04-04',   # 99.8% clear
    ('Delhi', 'December'): '2025-12-08',   # 99.9% clear
    ('Bangalore', 'April'):    '2025-04-14',   # 99.4% clear
    ('Bangalore', 'December'): '2025-12-10',   # 94.7% clear
    ('Hyderabad', 'April'):    '2025-04-06',   # 99.8% clear
    ('Hyderabad', 'December'): '2025-11-16',   # 97.1% clear
}

print("\nChecking staged inputs:")
missing = []
for c in CONFIGS:
    ok_csv, ok_geo = os.path.exists(c['csv']), os.path.exists(c['geojson'])
    print(f"  {c['short']} {c['month']:<9} csv [{'OK ' if ok_csv else 'MISSING'}]"
          f"  geojson [{'OK ' if ok_geo else 'MISSING'}]")
    missing += [p for p, ok in ((c['csv'], ok_csv), (c['geojson'], ok_geo)) if not ok]

for c in CONFIGS:
    os.makedirs(f"{OUT_ROOT}/{c['city']}/{c['month']}_Models", exist_ok=True)

PROTECT_EXISTING = False

def safe_path(path):
    if not PROTECT_EXISTING or not os.path.exists(path):
        return path
    stem, ext = os.path.splitext(path)
    n = 2
    while os.path.exists(f"{stem}_v{n}{ext}"):
        n += 1
    return f"{stem}_v{n}{ext}"

print(f"\nOutput tree ready under {OUT_ROOT}")
if missing:
    print(f"\n!! {len(set(missing))} input file(s) missing.")
else:
    print(f"\nAll {len(CONFIGS)} configs staged. Night-only: {NIGHT_ONLY}. "
          f"Window: {HP['window_size']}, horizon: {HP['horizon']}.")

In [ ]:
# ============================================================
# 3. IMPORTS + DETERMINISM
# ============================================================
import os, random, time, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')                 # headless: figures are saved, not shown
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats               # Spearman, used in the LST validation
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, PairNorm      # drop knn_graph
from torch_geometric.utils import dropout_adj, to_scipy_sparse_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")
if device.type == 'cuda':
    print(f"gpu   : {torch.cuda.get_device_name(0)}")
else:
    print("!! CPU only -- training will be slow.")


# ---- determinism ------------------------------------------------
# NOTE for the paper: set_seed() does NOT make GPU runs bit-reproducible.
# cuDNN's LSTM backward and the scatter reductions inside GATConv are
# both nondeterministic on CUDA, so two runs at the SAME seed differ
# slightly. That matters for how the ±0.185°C noise floor is described:
# it bounds seed variance AND run-to-run nondeterminism together, not
# seed variance alone. Either report it that way, or set
# DETERMINISTIC = True below and re-measure.
#
# DETERMINISTIC costs speed and still cannot remove all scatter
# nondeterminism -- it narrows the gap rather than closing it.
DETERMINISTIC = False

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True     # faster, nondeterministic


def knn_graph(pos, k, loop=False):
    """Pure-torch k-NN graph. Replaces torch_cluster.knn_graph.

    Returns edge_index [2, N*k] with flow source_to_target, matching
    PyG's convention: edge_index[0] = neighbour, edge_index[1] = node.

    pos is in EPSG:32643 metres (see load_nodes in cell 7), so cdist
    gives true metric distances. If pos were left in degrees the
    neighbourhoods would be latitude-distorted.
    """
    if not torch.isfinite(pos).all():
        raise ValueError("non-finite node positions -- geojson CRS was "
                         "mishandled upstream; see load_nodes() in cell 7")
    d = torch.cdist(pos, pos)                       # [N, N]
    if not loop:
        d.fill_diagonal_(float('inf'))
    idx = d.topk(k, largest=False).indices           # [N, k]
    N = pos.shape[0]
    dst = torch.arange(N, device=pos.device).repeat_interleave(k)
    src = idx.reshape(-1)
    return torch.stack([src, dst], dim=0)


# ---- self-test --------------------------------------------------
_p = torch.randn(100, 2, device=device)
_ei = knn_graph(_p, k=8, loop=False)
assert _ei.shape == (2, 800), _ei.shape
assert _ei[1].bincount().unique().tolist() == [8]     # every node has k=8
assert not (_ei[0] == _ei[1]).any()                   # no self-loops
print("knn_graph self-test passed")

## 3 · Dataset and models

In [ ]:
# ============================================================
# 4. DATASET + MODELS   (MODIS night LST)
# ------------------------------------------------------------
# FEATURE_COLS index map (cell 7). The slices below depend on this
# ORDER, not on the names -- reordering FEATURE_COLS would feed the
# wrong features to the wrong head with no error anywhere:
#    0  t2m       \
#    1  u10        |
#    2  v10        |- atmospheric, slice 0:6 -> physics_head
#    3  ssr        |
#    4  str        |
#    5  slhf      /
#    6  NDVI      \
#    7  NDBI       |
#    8  bld_height |- morphological, slice 6:11 -> residual_head
#    9  bld_cover  |
#   10  no2       /
#
# CHANGES FROM THE ERA5 VERSION:
#
# 1. THE TARGET IS A SEPARATE TENSOR. Previously the target was
#    feature 0 (temperature_2m), so the model saw its own history.
#    Now the target is MODIS night LST and is NOT among the inputs.
#    That is deliberate: given past LST the model leans on
#    persistence and never has to use morphology, which is precisely
#    how the ERA5 design produced spatial skill that turned out to be
#    the model recovering which coarse grid cell a node sat in.
#
# 2. horizon=0 -- ESTIMATION, NOT FORECASTING. The target is LST at
#    the LAST step of the window. The question is "given tonight's
#    regional atmospheric state and this cell's urban form, what is
#    tonight's surface temperature", and the LSTM's role is thermal
#    inertia: how the preceding nights' conditions carry forward.
#
# 3. atmos slice is 0:6, not 1:6, since t2m is now a predictor.
#    physics_head therefore takes lstm_hidden + 6 inputs.
# ============================================================

class PhyResDataset(Dataset):
    """Every sample is one time window over ALL nodes. `node_mask`
    selects which nodes contribute to the loss, so train/val/test
    differ by LOCATION rather than by date. Held-out nodes still
    participate in message passing -- their features are visible,
    their targets are not.

    `observed` flags cells that were really retrieved rather than
    interpolated, so the loss can ignore imputed targets."""
    def __init__(self, x_norm, x_raw, y, observed, base_edge_index, pos,
                 node_mask, window_size=5, horizon=0, use_pruning=True):
        self.x_norm, self.x_raw, self.y = x_norm, x_raw, y
        self.observed = observed
        self.base_edge_index, self.pos = base_edge_index, pos
        self.node_mask = node_mask
        self.window_size, self.horizon = window_size, horizon
        self.use_pruning = use_pruning
        self.T = x_norm.shape[1]
        self.num_samples = self.T - window_size - horizon + 1
        if self.num_samples < 1:
            raise ValueError("window longer than the series")

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        last = idx + self.window_size - 1
        x_window = self.x_norm[:, idx: idx + self.window_size, :]
        tgt = last + self.horizon
        y_target = self.y[:, tgt]
        # only score nodes in this split whose target was really observed
        loss_mask = self.node_mask & self.observed[:, tgt]

        if self.use_pruning:
            u = self.x_raw[:, last, 1].mean(); v = self.x_raw[:, last, 2].mean()
            wind = torch.stack([u, v])
            vecs = self.pos[self.base_edge_index[1]] - self.pos[self.base_edge_index[0]]
            ei = self.base_edge_index[:, (vecs * wind).sum(dim=1) >= 0]
        else:
            ei = self.base_edge_index

        return Data(x=x_window, edge_index=ei, y=y_target,
                    loss_mask=loss_mask,
                    raw_wind=self.x_raw[:, last, 1:3],
                    raw_last=self.x_raw[:, last, :])


class RobustSAGATLayer(nn.Module):
    """Attention over neighbours, with the node's own features carried
    past the aggregation by `self_linear`.

    That residual path is not incidental. On this data plain graph
    attention degrades about 0.10 R2 per layer and falls below zero by
    depth 6 -- worse than predicting the mean -- while the skip variant
    stays flat across depths 1 to 6.
    """
    def __init__(self, in_channels, out_channels, heads=1, dropout=0.2):
        super().__init__()
        self.neighbor_gat = GATConv(in_channels, out_channels, heads=heads,
                                    add_self_loops=False, concat=False,
                                    dropout=dropout)
        self.self_linear = nn.Linear(in_channels, out_channels)
        self.fusion = nn.Linear(out_channels * 2, out_channels)
        self.norm = PairNorm(scale=1.0)
        self.dropout_prob = dropout

    def forward(self, x, edge_index, batch=None):
        ei, _ = dropout_adj(edge_index, p=self.dropout_prob, training=self.training)
        neighbor_emb = self.neighbor_gat(x, ei)
        self_emb = self.self_linear(x)
        out = torch.cat([self_emb, neighbor_emb], dim=-1)
        out = F.leaky_relu(self.fusion(out), negative_slope=0.2)
        return self.norm(out + self_emb, batch)


def _vectorised_spatial(layer, x_seq, edge_index):
    """One batched pass over all timesteps instead of a Python loop.

    Builds a block-diagonal graph of T disconnected copies. `bvec`
    keeps PairNorm normalising WITHIN each timestep; without it the
    timesteps would be normalised jointly, which silently changes the
    model.
    """
    N, T, _ = x_seq.shape
    dev = x_seq.device
    x_flat = x_seq.permute(1, 0, 2).reshape(T * N, -1)
    E = edge_index.shape[1]
    offsets = (torch.arange(T, device=dev) * N).repeat_interleave(E)
    ei_big = edge_index.repeat(1, T) + offsets
    bvec = torch.arange(T, device=dev).repeat_interleave(N)
    emb = layer(x_flat, ei_big, bvec)
    return emb.view(T, N, -1).permute(1, 0, 2).contiguous()


class PhyRes(nn.Module):
    """Dual head: physics (atmospheric baseline) + residual (local
    morphological increment). Their sum is the predicted night LST.

    The decomposition only means something because the target now has
    intra-urban structure to divide: MODIS night LST varies 1.06-1.31 C
    across nodes, against 0.09-0.23 C for the ERA5 target this replaced.
    """
    def __init__(self, num_features, spatial_dim, lstm_hidden, dropout=0.2,
                 n_atmos=6, n_urban=5):
        super().__init__()
        self.n_atmos, self.n_urban = n_atmos, n_urban
        self.spatial_layer = RobustSAGATLayer(num_features, spatial_dim,
                                              dropout=dropout)
        self.temporal_lstm = nn.LSTM(spatial_dim, lstm_hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.physics_head = nn.Sequential(
            nn.Linear(lstm_hidden + n_atmos, lstm_hidden), nn.LayerNorm(lstm_hidden),
            nn.LeakyReLU(0.2), nn.Linear(lstm_hidden, 1))
        self.residual_head = nn.Sequential(
            nn.Linear(lstm_hidden + n_urban, lstm_hidden // 2),
            nn.LayerNorm(lstm_hidden // 2),
            nn.LeakyReLU(0.2), nn.Linear(lstm_hidden // 2, 1))
        self.res_scale = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x_seq, edge_index):
        spatial_seq = _vectorised_spatial(self.spatial_layer, x_seq, edge_index)
        lstm_out, _ = self.temporal_lstm(spatial_seq)
        last_state = self.dropout(lstm_out[:, -1, :])
        last_hour = x_seq[:, -1, :]
        atmos = last_hour[:, :self.n_atmos]                    # 0:6
        urban = last_hour[:, self.n_atmos:self.n_atmos + self.n_urban]   # 6:11
        t_phys = self.physics_head(torch.cat([last_state, atmos], -1)).squeeze(-1)
        t_res = self.residual_head(torch.cat([last_state, urban], -1)).squeeze(-1) \
            * self.res_scale
        return t_phys + t_res, t_phys, t_res

    def get_embeddings(self, x_seq, edge_index):
        spatial_seq = _vectorised_spatial(self.spatial_layer, x_seq, edge_index)
        lstm_out, _ = self.temporal_lstm(spatial_seq)
        return lstm_out[:, -1, :]


class VanillaBaselineModel(nn.Module):
    """Single head, no decomposition. Returns zeros for t_res so the
    figure functions can treat it uniformly."""
    def __init__(self, num_features, spatial_dim, lstm_hidden, dropout=0.2):
        super().__init__()
        self.spatial_layer = RobustSAGATLayer(num_features, spatial_dim,
                                              dropout=dropout)
        self.temporal_lstm = nn.LSTM(spatial_dim, lstm_hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden), nn.LayerNorm(lstm_hidden),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(lstm_hidden, 1))

    def forward(self, x_seq, edge_index):
        spatial_seq = _vectorised_spatial(self.spatial_layer, x_seq, edge_index)
        lstm_out, _ = self.temporal_lstm(spatial_seq)
        last_state = self.dropout(lstm_out[:, -1, :])
        t_total = self.output_head(last_state).squeeze(-1)
        return t_total, t_total, torch.zeros_like(t_total)

    def get_embeddings(self, x_seq, edge_index):
        spatial_seq = _vectorised_spatial(self.spatial_layer, x_seq, edge_index)
        lstm_out, _ = self.temporal_lstm(spatial_seq)
        return lstm_out[:, -1, :]


class NoGraphModel(nn.Module):
    """Same inputs and depth, no message passing -- a per-node MLP+LSTM.

    Worth having as a first-class arm rather than an afterthought. On
    the Landsat version of this task, ordinary least squares on the
    five morphological features reached R2 0.60-0.75 and a random
    forest matched or beat the graph on transfer. If this ties the
    graph model, the graph is not earning its place and the paper
    should say so.
    """
    def __init__(self, num_features, spatial_dim, lstm_hidden, dropout=0.2,
                 n_atmos=6, n_urban=5):
        super().__init__()
        self.n_atmos, self.n_urban = n_atmos, n_urban
        self.encoder = nn.Sequential(
            nn.Linear(num_features, spatial_dim), nn.LeakyReLU(0.2),
            nn.Linear(spatial_dim, spatial_dim))
        self.temporal_lstm = nn.LSTM(spatial_dim, lstm_hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.physics_head = nn.Sequential(
            nn.Linear(lstm_hidden + n_atmos, lstm_hidden), nn.LayerNorm(lstm_hidden),
            nn.LeakyReLU(0.2), nn.Linear(lstm_hidden, 1))
        self.residual_head = nn.Sequential(
            nn.Linear(lstm_hidden + n_urban, lstm_hidden // 2),
            nn.LayerNorm(lstm_hidden // 2),
            nn.LeakyReLU(0.2), nn.Linear(lstm_hidden // 2, 1))
        self.res_scale = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x_seq, edge_index):
        h = self.encoder(x_seq)
        lstm_out, _ = self.temporal_lstm(h)
        last_state = self.dropout(lstm_out[:, -1, :])
        last_hour = x_seq[:, -1, :]
        atmos = last_hour[:, :self.n_atmos]
        urban = last_hour[:, self.n_atmos:self.n_atmos + self.n_urban]
        t_phys = self.physics_head(torch.cat([last_state, atmos], -1)).squeeze(-1)
        t_res = self.residual_head(torch.cat([last_state, urban], -1)).squeeze(-1) \
            * self.res_scale
        return t_phys + t_res, t_phys, t_res

    def get_embeddings(self, x_seq, edge_index):
        h = self.encoder(x_seq)
        lstm_out, _ = self.temporal_lstm(h)
        return lstm_out[:, -1, :]

## 4 · Training

In [ ]:
# ============================================================
# 5. TRAINING   (MODIS night LST)
# ------------------------------------------------------------
# THREE CHANGES FROM THE ERA5 VERSION.
#
# 1. THRESHOLDS RECALIBRATED FOR 1 km NODES. Averaging over 1 km
#    instead of 500 m compresses bld_cover: the max fell from
#    0.59-0.69 to 0.50-0.60 and medians sit at 0.16-0.19. The old
#    COVER_T = 0.40 would now fire on only a few percent of Bengaluru
#    nodes. Defaults below are a starting point -- run
#    report_mask_coverage() on each config and adjust so the selected
#    fraction is comparable across cities.
#
# 2. MARGINS ARE NOW IN DEGREES. Previously push used
#    relu(0.8 - t_res), i.e. "at least 0.8 standard deviations",
#    whose physical meaning changed silently whenever the target
#    changed. The effect size is now known -- night LST varies
#    1.06-1.31 C across nodes and correlates +0.44 to +0.69 with
#    bld_cover -- so the constraint can be stated in C and converted
#    using temp_std. That makes it reportable: "the prior asserts
#    dense nodes run at least 1 C above the atmospheric baseline"
#    is a claim a reviewer can evaluate; "0.8 normalised units" is not.
#
#    composite_loss therefore takes temp_std. Update the call sites.
#
# 3. THE PRIOR NOW POINTS THE RIGHT WAY. Against daytime Landsat,
#    corr(bld_cover, LST) ran -0.04 to +0.50 and the push term was
#    contradicted in half the configs. Against night MODIS it is
#    +0.44 to +0.69 in all six. The constraint is no longer fighting
#    the validation data.
#
# gamma STILL NEEDS RE-TUNING. Mask normalisation is retained, and
# both the target scale and the mask fractions have changed. Run
# loss_breakdown() before trusting any sweep result.
# ============================================================

# ---- push-pull configuration -------------------------------------
COVER_T = 0.30        # raw building coverage fraction, 1 km nodes
NDVI_T  = 0.40        # raw NDVI
PUSH_MARGIN_C = 1.0   # dense nodes: at least this far ABOVE the baseline
PULL_MARGIN_C = 0.5   # vegetated nodes: at least this far BELOW
SPARSITY_W = 0.005
NORMALIZE_BY_MASK = True

COVER_IDX, NDVI_IDX = 9, 6      # positions in FEATURE_COLS


def composite_loss(model_out, batch, criterion, alpha, gamma, temp_std,
                   report=None):
  final_pred, t_phys, t_res = model_out
  m = batch.loss_mask
  if m.sum() < 1:
      return (torch.zeros((), device=batch.x.device, requires_grad=True),
              torch.zeros((), device=batch.x.device))
  mse_loss = criterion(final_pred[m], batch.y[m])
  mse_loss = criterion(final_pred, batch.y)

  row, col = batch.edge_index
  if row.numel() == 0:
      # mean() over zero elements is NaN and 0.0 * NaN is still NaN,
      # which poisoned the LR scheduler in the no-graph ablation.
      physics_loss = torch.zeros((), device=batch.x.device)
  else:
      u_w, v_w = batch.raw_wind[row, 0], batch.raw_wind[row, 1]
      wind_speed = torch.sqrt(u_w ** 2 + v_w ** 2 + 1e-6)
      physics_loss = torch.mean(
          torch.abs(t_phys[row] - t_phys[col]) / (wind_speed + 1.0))

  # Physical thresholds on raw_last, not z-scores on batch.x.
  # raw_last is never touched by the feature mask, so a
  # morphology-dropped ablation cannot silently disable the prior.
  building = batch.raw_last[:, COVER_IDX]
  ndvi = batch.raw_last[:, NDVI_IDX]
  urban_mask = (building > COVER_T).float()
  veg_mask = (ndvi > NDVI_T).float()
  neutral = 1.0 - (urban_mask + veg_mask).clamp(0, 1)

  # Margins in degrees -> normalised units, so the constraint means
  # the same thing whatever the target's scale.
  push_m = PUSH_MARGIN_C / max(temp_std, 1e-6)
  pull_m = PULL_MARGIN_C / max(temp_std, 1e-6)

  if NORMALIZE_BY_MASK:
      # Mean violation AMONG masked nodes, so constraint strength does
      # not scale with how many nodes each city happens to have above
      # threshold. clamp guards a config where a mask selects nothing.
      push = (F.relu(push_m - t_res) * urban_mask).sum() / urban_mask.sum().clamp(min=1.0)
      pull = (F.relu(t_res + pull_m) * veg_mask).sum() / veg_mask.sum().clamp(min=1.0)
      sparsity = (t_res ** 2 * neutral).sum() / neutral.sum().clamp(min=1.0)
  else:
      push = torch.mean(F.relu(push_m - t_res) * urban_mask)
      pull = torch.mean(F.relu(t_res + pull_m) * veg_mask)
      sparsity = torch.mean((t_res ** 2) * neutral)

  total = mse_loss + alpha * physics_loss + gamma * (push + pull) \
      + SPARSITY_W * sparsity

  if report is not None:
      report['huber'] += mse_loss.item()
      report['physics_w'] += (alpha * physics_loss).item()
      report['pushpull_w'] += (gamma * (push + pull)).item()
      report['sparsity_w'] += (SPARSITY_W * sparsity).item()
      report['total'] += total.item()
      report['n'] += 1
  return total, mse_loss


def report_mask_coverage(loader, tag, temp_std=None):
    """What the masks select, and how normalisation rescales each term.

    These fractions are the operative definition of 'built-up' and
    'vegetated' used by the loss and belong in the paper. Aim for a
    comparable fraction across cities: a mask firing on 3% of nodes in
    one city and 25% in another is not the same constraint.
    """
    batch = next(iter(loader))
    b, v = batch.raw_last[:, COVER_IDX], batch.raw_last[:, NDVI_IDX]
    u_m, v_m = (b > COVER_T), (v > NDVI_T)
    n = b.numel()
    nu, nv = int(u_m.sum()), int(v_m.sum())
    print(f"    [{tag}] cover>{COVER_T}: {nu}/{n} ({100*nu/n:.1f}%)"
          f"   NDVI>{NDVI_T}: {nv}/{n} ({100*nv/n:.1f}%)"
          f"   overlap: {int((u_m & v_m).sum())}")
    print(f"           bld_cover  median {float(b.median()):.3f}  "
          f"p75 {float(b.quantile(0.75)):.3f}  max {float(b.max()):.3f}")
    print(f"           NDVI       median {float(v.median()):.3f}  "
          f"p75 {float(v.quantile(0.75)):.3f}  max {float(v.max()):.3f}")
    if temp_std:
        print(f"           margins    push {PUSH_MARGIN_C} C = "
              f"{PUSH_MARGIN_C/temp_std:.3f} norm   pull {PULL_MARGIN_C} C = "
              f"{PULL_MARGIN_C/temp_std:.3f} norm")
    if NORMALIZE_BY_MASK:
        print(f"           gamma multiplier: push x{n/max(nu,1):.1f}, "
              f"pull x{n/max(nv,1):.1f}")
    if nu == 0:
        print(f"    !! urban_mask empty -- push term dead. "
              f"max cover {float(b.max()):.3f}, lower COVER_T.")
    if nv == 0:
        print(f"    !! veg_mask empty -- pull term dead. "
              f"max NDVI {float(v.max()):.3f}, lower NDVI_T.")


def loss_breakdown(model, loader, alpha, gamma, temp_std, tag):
    """Each term's share of the objective. Use this to set gamma:
    both the target scale and the mask fractions changed, so any gamma
    carried over from the ERA5 runs is meaningless."""
    crit = nn.HuberLoss(delta=1.5)
    rep = dict(huber=0., physics_w=0., pushpull_w=0., sparsity_w=0.,
               total=0., n=0)
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            composite_loss(model(batch.x, batch.edge_index), batch, crit,
                           alpha, gamma, temp_std, report=rep)
    n, tot = rep['n'], rep['total']
    print(f"\n  loss composition [{tag}]  (alpha={alpha}, gamma={gamma})")
    for k in ('huber', 'physics_w', 'pushpull_w', 'sparsity_w'):
        print(f"    {k:<12} {rep[k]/n:>10.6f}   {100*rep[k]/tot:>6.2f}% of total")
    print(f"    {'TOTAL':<12} {tot/n:>10.6f}")
    return {k: rep[k] / n for k in rep if k != 'n'}


def train_model(model, train_loader, val_loader, epochs, lr, alpha, gamma,
                weight_decay, clip, tag, temp_std=1.0, patience=40,
                verbose_every=25):
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5,
                                                 patience=8)
    criterion = nn.HuberLoss(delta=1.5)
    t0 = time.time()
    best_rmse, best_state, best_epoch, since = float('inf'), None, -1, 0

    for epoch in range(epochs):
        model.train()
        tr_huber, nb = 0.0, 0
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            loss, huber = composite_loss(model(batch.x, batch.edge_index),
                                         batch, criterion, alpha, gamma, temp_std)
            loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
            opt.step()
            tr_huber += huber.item(); nb += 1

        model.eval()
        va_loss, sq_err, n_elem = 0.0, 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index)
                loss, _ = composite_loss(out, batch, criterion, alpha, gamma,
                                         temp_std)
                va_loss += loss.item()
                err = (out[0][batch.loss_mask] - batch.y[batch.loss_mask]) * temp_std
                sq_err += err.pow(2).sum().item()
                n_elem += err.numel()

        va_loss /= max(len(val_loader), 1)
        va_rmse = (sq_err / max(n_elem, 1)) ** 0.5
        sched.step(va_loss)

        if va_rmse < best_rmse - 1e-4:
            best_rmse, best_epoch, since = va_rmse, epoch + 1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            # Splits are short now (a 14-day window gives 13-26 night
            # timesteps, so only a handful of windows per split).
            # Early stopping matters more than it did with 336 hours.
            if since >= patience:
                break

        if (epoch + 1) % verbose_every == 0 or epoch == 0:
            print(f"    [{tag}] epoch {epoch+1:>3}/{epochs} | "
                  f"train Huber {tr_huber/max(nb,1):.4f} | "
                  f"val {va_loss:.4f} | val RMSE {va_rmse:.3f}°C")

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"    [{tag}] {(time.time()-t0)/60:.1f} min | best ep {best_epoch} "
          f"(val RMSE {best_rmse:.3f}°C)")
    return model, dict(best_epoch=best_epoch, best_val_rmse=best_rmse)


# ------------------------------------------------------------
# CALL SITES CHANGE — composite_loss now takes temp_std:
#
#     composite_loss(out, batch, crit, alpha, gamma, D['temp_std'])
#     loss_breakdown(m, L['val'], ALPHA, GAMMA, D['temp_std'], tag)
#     report_mask_coverage(L['train'], tag, D['temp_std'])
#
# train_model already received temp_std and passes it through.
# ------------------------------------------------------------

## 5 · Evaluation and figures

In [ ]:
# ============================================================
# 6. EVALUATION + FIGURES   (MODIS night LST)
# ------------------------------------------------------------
# CHANGES:
#  - the target is a separate tensor, so batch.y is night LST rather
#    than feature 0
#  - feature indices shifted: NDVI 6, NDBI 7, bld_height 8,
#    bld_cover 9, no2 10
#  - spatial skill is now reported as a first-class metric. Overall
#    R2 on this task is dominated by night-to-night variation shared
#    by every node; a model that tracks the city-wide mean and nothing
#    spatial still scores well. The per-timestep spatial anomaly
#    correlation is what says whether the graph is doing anything.
#  - Pearson and Spearman together, since cover-vs-LST relationships
#    were non-monotone in some daytime configs
# ============================================================

def diagnostic_eval(model, test_loader, temp_mean, temp_std, save_path, title):
    """Predicted vs observed night LST, on HELD-OUT BLOCKS only.

    batch.loss_mask selects nodes in this split whose target was really
    retrieved rather than interpolated. Without it every metric would
    span all nodes -- training blocks included -- and the reported R2
    would be part training accuracy.
    """
    model.eval()
    P, R, T = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            msk = batch.loss_mask
            if msk.sum() < 1:
                continue
            fp, tp, tr = model(batch.x, batch.edge_index)
            P.append((fp[msk] * temp_std + temp_mean).cpu().numpy())
            R.append((tr[msk] * temp_std).cpu().numpy())
            T.append((batch.y[msk] * temp_std + temp_mean).cpu().numpy())

    preds, targets, res = np.concatenate(P), np.concatenate(T), np.concatenate(R)
    rmse = float(np.sqrt(mean_squared_error(targets, preds)))
    mae = float(mean_absolute_error(targets, preds))
    r2 = 1 - float(((targets - preds) ** 2).sum()
                   / ((targets - targets.mean()) ** 2).sum())

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].scatter(targets, preds, alpha=0.3, s=4)
    lo, hi = targets.min(), targets.max()
    ax[0].plot([lo, hi], [lo, hi], 'r--', lw=2)
    ax[0].set_title(f"{title} — held-out night LST "
                    f"(RMSE {rmse:.3f}°C, R² {r2:.3f}, n={len(targets)})")
    ax[0].set_xlabel("observed MODIS night LST (°C)")
    ax[0].set_ylabel("predicted (°C)")
    ax[1].hist(res, bins=50, color='purple')
    ax[1].set_title("residual head output — local UHI increment")
    ax[1].set_xlabel("°C above/below the atmospheric baseline")
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    return dict(rmse=rmse, mae=mae, r2=r2, n_eval=int(len(targets)),
                res_max=float(res.max()), res_min=float(res.min()),
                res_mean=float(res.mean()), res_std=float(res.std())), \
        preds, res, targets


def spatial_skill(model, loader, temp_mean, temp_std, tag=""):
    """Does the model reproduce the spatial PATTERN, or only the
    night-to-night level?

    Held-out nodes only, with the anomaly taken relative to the mean
    over those same nodes -- so the statistic asks whether the model
    ranks UNSEEN locations correctly. Overall R2 cannot answer that:
    it is dominated by night-to-night variation every node shares.
    """
    model.eval()
    P, Y = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            msk = batch.loss_mask
            if msk.sum() < 5:
                continue
            fp, _, _ = model(batch.x, batch.edge_index)
            P.append(fp[msk].cpu().numpy()); Y.append(batch.y[msk].cpu().numpy())
    if not P:
        return dict(spatial_r_mean=np.nan, spatial_r_median=np.nan,
                    spatial_spread_ratio=np.nan, actual_spatial_sd=np.nan,
                    pred_spatial_sd=np.nan)

    # node membership varies between windows (a node drops out on nights
    # its retrieval was cloudy), so this stays ragged rather than stacked
    rs, pa_all, ya_all = [], [], []
    for p, y in zip(P, Y):
        pa, ya = p - p.mean(), y - y.mean()
        if pa.std() > 1e-9 and ya.std() > 1e-9:
            rs.append(np.corrcoef(ya, pa)[0, 1])
        pa_all.append(pa); ya_all.append(ya)
    rs = np.array(rs) if rs else np.array([np.nan])
    pa_all, ya_all = np.concatenate(pa_all), np.concatenate(ya_all)

    ratio = float(pa_all.std() / ya_all.std()) if ya_all.std() > 0 else np.nan
    out = dict(spatial_r_mean=float(np.nanmean(rs)),
               spatial_r_median=float(np.nanmedian(rs)),
               spatial_spread_ratio=ratio,
               actual_spatial_sd=float(ya_all.std() * temp_std),
               pred_spatial_sd=float(pa_all.std() * temp_std))
    print(f"    [{tag}] spatial r {out['spatial_r_mean']:+.3f} "
          f"(median {out['spatial_r_median']:+.3f}) | "
          f"observed spatial sd {out['actual_spatial_sd']:.2f}°C | "
          f"predicted {out['pred_spatial_sd']:.2f}°C (ratio {ratio:.2f})")
    if out['spatial_r_mean'] < 0.2:
        print("    !! little spatial skill on held-out blocks")
    if ratio < 0.5:
        print("    !! predictions flatter than reality -- hedging to the mean")
    return out


def residual_correlations(model, loader, save_path, title):
    """t_res against each urban feature, Pearson and Spearman.

    Features come from raw_last (physical units). Pearson is unchanged
    by z-scoring; Spearman is the one to read where a relationship may
    turn inside its range.
    """
    model.eval()
    names = {'Traffic (NO2 proxy)': 10, 'Building coverage': 9,
             'Building height': 8, 'Built-up index (NDBI)': 7,
             'Vegetation (NDVI)': 6}
    cols = {k: [] for k in ['Residual heat (t_res)'] + list(names)}
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            msk = batch.loss_mask
            if msk.sum() < 1:
                continue
            _, _, t_res = model(batch.x, batch.edge_index)
            cols['Residual heat (t_res)'].append(t_res[msk].cpu().numpy())
            for label, idx in names.items():
                cols[label].append(batch.raw_last[msk, idx].cpu().numpy())

    df = pd.DataFrame({k: np.concatenate(v) for k, v in cols.items()})

    if df['Residual heat (t_res)'].std() < 1e-12:
        plt.figure(figsize=(6, 5))
        plt.text(0.5, 0.5, "No residual head\n(correlation undefined)",
                 ha='center', va='center', fontsize=13)
        plt.axis('off'); plt.title(title, fontweight='bold')
        plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close()
        return {k: float('nan') for k in names}

    t = df['Residual heat (t_res)'].values
    mat = pd.DataFrame({
        'Pearson': {k: float(np.corrcoef(df[k].values, t)[0, 1]) for k in names},
        'Spearman': {k: float(stats.spearmanr(df[k].values, t).statistic)
                     for k in names}})

    plt.figure(figsize=(7, 5))
    sns.heatmap(mat, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".3f",
                cbar_kws={'label': 'correlation'})
    plt.title(f"{title}\nurban form vs residual head", fontweight='bold')
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    for k in names:
        p, s = mat.loc[k, 'Pearson'], mat.loc[k, 'Spearman']
        if abs(p - s) > 0.15:
            print(f"    note: {k} Pearson {p:+.3f} vs Spearman {s:+.3f} "
                  f"-- non-linear, do not quote Pearson alone")
    return {k: float(mat.loc[k, 'Pearson']) for k in names}


def collapse_check(model, loader, save_path, title, threshold=0.99):
    """Fraction of distinct node pairs with near-identical embeddings."""
    model.eval()
    rates, last_mat = [], None
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            emb = F.normalize(model.get_embeddings(batch.x, batch.edge_index),
                              p=2, dim=1)
            sim = emb @ emb.T
            n = sim.shape[0]
            iu = torch.triu_indices(n, n, offset=1, device=sim.device)
            rates.append((sim[iu[0], iu[1]] > threshold).float().mean().item() * 100)
            last_mat = sim.cpu().numpy()

    rate = float(np.mean(rates))
    plt.figure(figsize=(7, 6))
    sns.heatmap(last_mat[:300, :300], cmap='magma', vmin=-1, vmax=1,
                cbar_kws={'label': 'cosine similarity'})
    plt.title(f"{title}\nnode collapse @>{threshold}: {rate:.2f}%  "
              f"(first 300 nodes)", fontweight='bold')
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    return rate


def city_residual_map(model, loader, gdf, temp_std, save_path, title,
                      average=True):
    """Residual heat per node, averaged over the test set by default.

    The time-averaged field is the persistent pattern -- the planning
    product -- and is the same quantity the LST check validates. A
    single window is a noisy sample of it.
    """
    model.eval()
    acc, n = None, 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, _, t_res = model(batch.x, batch.edge_index)
            v = (t_res * temp_std).cpu().numpy()
            acc = v if acc is None else acc + v
            n += 1
            if not average:
                break
    residual = acc / n

    plot_gdf = gdf.copy()
    if len(plot_gdf) != len(residual):
        print(f"    !! gdf has {len(plot_gdf)} rows, model returned "
              f"{len(residual)} -- skipping {os.path.basename(save_path)}")
        return
    plot_gdf['residual_c'] = residual

    lim = float(np.abs(residual).max())
    fig, ax = plt.subplots(figsize=(10, 10))
    plot_gdf.to_crs(epsg=3857).plot(column='residual_c', cmap='coolwarm',
                                    vmin=-lim, vmax=lim, alpha=0.75, ax=ax,
                                    legend=True,
                                    legend_kwds={'label': "residual heat (°C)",
                                                 'shrink': 0.6})
    try:
        import contextily as ctx
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
    except Exception as e:
        print(f"    (basemap unavailable: {e})")
    ax.set_title(title + (f" (mean of {n} test windows)" if average
                          else " (single window)"), fontweight='bold')
    ax.set_axis_off()
    plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close(fig)


def adjacency_figure(base_edge_index, save_path, title):
    adj = to_scipy_sparse_matrix(base_edge_index.cpu())
    plt.figure(figsize=(8, 8))
    plt.spy(adj, markersize=0.3, color='blue')
    plt.title(title, fontsize=14)
    plt.xlabel("node id"); plt.ylabel("node id")
    plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close()

## 6 · Spatially blocked splits

5 × 5 blocks, 60/15/25 by block. Normalisation statistics come from training nodes only. Loss and metrics are masked to genuinely retrieved cells.

In [ ]:
# ============================================================
# 7. DATA PREPARATION — SPATIALLY BLOCKED SPLITS
# ------------------------------------------------------------
# WHY THIS CHANGED
#
# The temporal split was the wrong axis. With horizon=0 the model
# estimates tonight's LST from tonight's atmospherics and this cell's
# urban form -- a SPATIAL estimation task. Splitting by time asks
# "can it predict unseen nights"; the question a UHI map poses is
# "can it predict unseen locations".
#
# The arithmetic changes completely. Delhi April has 24 night passes,
# so a 5-step window yields 20 windows. Split by TIME, those 20 are
# divided three ways and validation ends up with one or two. Split by
# SPACE, all 20 windows are available to every split and the nodes
# are divided instead -- roughly 520 training nodes x 20 windows.
# Same data, an order of magnitude more supervision.
#
# Blocks, not random nodes. Neighbouring nodes are strongly
# autocorrelated and the graph reads a node's neighbours directly, so
# a random node split scores the model on near-copies of its training
# data. On the Landsat version that inflated the graph model's
# apparent advantage over regression by about 3.5x.
#
# The model still sees held-out nodes' FEATURES during message
# passing; only their targets are hidden. That is standard
# transductive node regression and must be stated. Cross-city
# transfer is the stricter test and has no such advantage.
#
# CHANGES NEEDED IN CELLS 4 AND 5 ARE AT THE BOTTOM OF THIS FILE.
# ============================================================

FEATURE_COLS = ['t2m', 'u10', 'v10', 'ssr', 'str', 'slhf',
                'NDVI', 'NDBI', 'bld_height', 'bld_cover', 'traffic_no2_proxy']
TARGET_COL = 'lst_modis'
TIME_COL, ID_COL = 'timestamp', 'id'
TARGET_CRS = 32643
N_BLOCKS = 5          # 5x5 grid of spatial blocks

_EXPECTED_ORDER = list(FEATURE_COLS)
assert FEATURE_COLS == _EXPECTED_ORDER


def _align_ids(s):
    try:
        return s.astype('int64')
    except (ValueError, TypeError):
        return s.astype(str)


def load_nodes(path, target_crs=TARGET_CRS):
    """GeoJSON carries no CRS member, so geopandas assumes 4326. The
    grid was built with coveringGrid in EPSG:32643, so the file holds
    METRES and to_crs() on it returns inf. Magnitude is the
    unambiguous test: no geographic coordinate exceeds 180."""
    g = gpd.read_file(path)
    b = g.total_bounds
    if max(abs(b[0]), abs(b[2]), abs(b[1]), abs(b[3])) > 180:
        g = g.set_crs(target_crs, allow_override=True)
    else:
        if g.crs is None:
            g = g.set_crs(4326)
        g = g.to_crs(epsg=target_crs)
    return g


def prepare(cfg, hp, fold=0):
    """fold selects which spatial blocks are held out (0..N_FOLDS-1),
    so the same call can drive a blocked cross-validation."""
    df = pd.read_csv(cfg['csv'])
    gdf = load_nodes(cfg['geojson'])

    for col in (ID_COL, TIME_COL, TARGET_COL, 'pass'):
        if col not in df.columns:
            raise KeyError(f"'{col}' missing from {os.path.basename(cfg['csv'])}")
    miss = [c for c in FEATURE_COLS if c not in df.columns]
    if miss:
        raise KeyError(f"{miss} missing from {os.path.basename(cfg['csv'])}")

    df[ID_COL] = _align_ids(df[ID_COL])
    gdf[ID_COL] = _align_ids(gdf[ID_COL])
    if df[ID_COL].dtype != gdf[ID_COL].dtype:
        df[ID_COL] = df[ID_COL].astype(str); gdf[ID_COL] = gdf[ID_COL].astype(str)

    if np.issubdtype(df[TIME_COL].dtype, np.number):
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], unit='ms')
    else:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL])

    if NIGHT_ONLY:
        df = df[df['pass'].str.contains('night', na=False)]
    if QC_STRICT and 'qc_strict' in df.columns:
        df = df[df['qc_strict'] == 1]

    # ---- keep every night pass -----------------------------------
    # With a spatial split there is no reason to discard passes for
    # partial coverage: a cloudy node on a given night is one missing
    # cell, not a reason to drop the night for every node. Only nodes
    # too sparse to interpolate honestly are removed.
    times = np.sort(df[TIME_COL].unique())
    all_nodes = np.sort(df[ID_COL].unique())
    valid = df[df[TARGET_COL].notna()]
    ncov = valid.groupby(ID_COL)[TIME_COL].nunique() / len(times)
    keep_n = ncov[ncov >= MIN_NODE_COVERAGE].index
    print(f"  {len(times)} night passes | nodes {len(all_nodes)} -> "
          f"{len(keep_n)} (>= {MIN_NODE_COVERAGE:.0%} of passes observed)")
    if len(keep_n) < 200:
        print("  !! fewer than 200 nodes survive; blocked splits will be thin")

    df = df[df[ID_COL].isin(keep_n)]
    gdf = gdf[gdf[ID_COL].isin(keep_n)].sort_values(ID_COL).reset_index(drop=True)
    node_ids = np.sort(df[ID_COL].unique())
    if len(gdf) != len(node_ids) or not (gdf[ID_COL].values == node_ids).all():
        raise ValueError("node id mismatch between CSV and geojson")

    n_time = len(times)
    if n_time < hp['window_size'] + 2:
        raise ValueError(f"{n_time} passes cannot fill a "
                         f"{hp['window_size']}-step window")

    # ---- dense cube, then interpolate along each node's series ----
    cube = np.full((len(node_ids), n_time, len(FEATURE_COLS)), np.nan, np.float32)
    ymat = np.full((len(node_ids), n_time), np.nan, np.float32)
    ni = {v: i for i, v in enumerate(node_ids)}
    ti = {v: i for i, v in enumerate(times)}
    r = df[ID_COL].map(ni).values
    c = df[TIME_COL].map(ti).values
    cube[r, c, :] = df[FEATURE_COLS].values.astype(np.float32)
    ymat[r, c] = df[TARGET_COL].values.astype(np.float32)

    gap_frac = float(np.isnan(ymat).mean())
    print(f"  target matrix {ymat.shape} | {gap_frac:.1%} interpolated")
    if gap_frac > 0.5:
        print("  !! over half the target is interpolated. Any result here is")
        print("     substantially a statement about the interpolation.")

    def fill(a):
        out = a.copy()
        idx = np.arange(out.shape[1])
        for i in range(out.shape[0]):
            v = out[i]; m = ~np.isnan(v)
            if m.sum() == 0:
                raise ValueError(f"node {i} has no valid observations")
            out[i] = np.interp(idx, idx[m], v[m])
        return out

    for f in range(cube.shape[2]):
        cube[:, :, f] = fill(cube[:, :, f])
    ymat_filled = fill(ymat)
    observed = torch.tensor(~np.isnan(ymat), dtype=torch.bool)   # real vs imputed

    x_cont = torch.tensor(cube, dtype=torch.float)
    y_cont = torch.tensor(ymat_filled, dtype=torch.float)
    n_nodes = x_cont.shape[0]

    # ---- spatial blocks ------------------------------------------
    px = gdf.geometry.centroid.x.values
    py = gdf.geometry.centroid.y.values
    bx = np.floor((px - px.min()) / (np.ptp(px) + 1e-9) * N_BLOCKS).clip(0, N_BLOCKS - 1)
    by = np.floor((py - py.min()) / (np.ptp(py) + 1e-9) * N_BLOCKS).clip(0, N_BLOCKS - 1)
    block = (bx * N_BLOCKS + by).astype(int)
    ublocks = np.unique(block)

    rng = np.random.default_rng(hp['seed'])
    perm = rng.permutation(ublocks)
    n_te = max(1, int(len(perm) * 0.25))
    n_va = max(1, int(len(perm) * 0.15))
    te_b = np.roll(perm, fold * n_te)[:n_te]
    rest = np.setdiff1d(perm, te_b)
    va_b, tr_b = rest[:n_va], rest[n_va:]

    masks = {k: torch.tensor(np.isin(block, b), dtype=torch.bool)
             for k, b in (('train', tr_b), ('val', va_b), ('test', te_b))}
    print(f"  blocks {len(ublocks)} -> train {len(tr_b)} / val {len(va_b)} "
          f"/ test {len(te_b)} (fold {fold})")
    print(f"  nodes  train {int(masks['train'].sum())} / "
          f"val {int(masks['val'].sum())} / test {int(masks['test'].sum())}")
    n_windows = n_time - hp['window_size'] - hp['horizon'] + 1
    print(f"  {n_windows} windows, ALL available to every split")

    # ---- normalisation from TRAINING NODES only ------------------
    trm = masks['train']
    xs = x_cont[trm]
    means, stds = xs.mean(dim=(0, 1), keepdim=True), xs.std(dim=(0, 1), keepdim=True)
    x_norm = (x_cont - means) / (stds + 1e-6)
    ys = y_cont[trm]
    t_mean, t_std = ys.mean().item(), ys.std().item()
    y_norm = (y_cont - t_mean) / (t_std + 1e-6)
    print(f"  target: mean {t_mean:.2f} °C  std {t_std:.2f} °C")

    spatial_sd = float(np.nanmean(np.nanstd(ymat, axis=0)))
    print(f"  spatial sd {spatial_sd:.2f} °C  (this is what the graph must explain)")

    pos = torch.tensor(np.column_stack([px, py]), dtype=torch.float).to(device)
    if not torch.isfinite(pos).all():
        raise ValueError("non-finite node positions -- geojson CRS mishandled")
    base_edge_index = knn_graph(pos, k=hp['k_neighbors'], loop=False).to(device)

    x_norm, x_cont = x_norm.to(device), x_cont.to(device)
    y_norm = y_norm.to(device)
    observed = observed.to(device)
    masks = {k: v.to(device) for k, v in masks.items()}

    def loaders(pruned):
        out = {}
        for name in ('train', 'val', 'test'):
            ds = PhyResDataset(x_norm, x_cont, y_norm, observed,
                               base_edge_index, pos, masks[name],
                               window_size=hp['window_size'],
                               horizon=hp['horizon'], use_pruning=pruned)
            out[name] = DataLoader(ds, batch_size=1, shuffle=(name == 'train'))
        return out

    yc = y_cont.cpu().numpy()
    persist = float(np.sqrt(np.nanmean((yc[:, 1:] - yc[:, :-1]) ** 2)))
    print(f"  night-to-night persistence RMSE {persist:.2f} °C")

    return dict(gdf=gdf, n_nodes=n_nodes, n_time=n_time,
                node_ids=node_ids, times=times, block=block, masks=masks,
                temp_mean=t_mean, temp_std=t_std, gap_frac=gap_frac,
                spatial_sd=spatial_sd, persistence=persist, fold=fold,
                base_edge_index=base_edge_index,
                pruned=loaders(True), static=loaders(False))


# ============================================================
# CELL 4 — replace PhyResDataset with this
# ============================================================
# class PhyResDataset(Dataset):
#     """Every sample is one time window over ALL nodes. `node_mask`
#     selects which nodes contribute to the loss, so train/val/test
#     differ by LOCATION rather than by date. Held-out nodes still
#     participate in message passing -- their features are visible,
#     their targets are not.
#
#     `observed` flags cells that were really retrieved rather than
#     interpolated, so the loss can ignore imputed targets."""
#     def __init__(self, x_norm, x_raw, y, observed, base_edge_index, pos,
#                  node_mask, window_size=5, horizon=0, use_pruning=True):
#         self.x_norm, self.x_raw, self.y = x_norm, x_raw, y
#         self.observed = observed
#         self.base_edge_index, self.pos = base_edge_index, pos
#         self.node_mask = node_mask
#         self.window_size, self.horizon = window_size, horizon
#         self.use_pruning = use_pruning
#         self.T = x_norm.shape[1]
#         self.num_samples = self.T - window_size - horizon + 1
#         if self.num_samples < 1:
#             raise ValueError("window longer than the series")
#
#     def __len__(self):
#         return self.num_samples
#
#     def __getitem__(self, idx):
#         last = idx + self.window_size - 1
#         x_window = self.x_norm[:, idx: idx + self.window_size, :]
#         tgt = last + self.horizon
#         y_target = self.y[:, tgt]
#         # only score nodes in this split whose target was really observed
#         loss_mask = self.node_mask & self.observed[:, tgt]
#
#         if self.use_pruning:
#             u = self.x_raw[:, last, 1].mean(); v = self.x_raw[:, last, 2].mean()
#             wind = torch.stack([u, v])
#             vecs = self.pos[self.base_edge_index[1]] - self.pos[self.base_edge_index[0]]
#             ei = self.base_edge_index[:, (vecs * wind).sum(dim=1) >= 0]
#         else:
#             ei = self.base_edge_index
#
#         return Data(x=x_window, edge_index=ei, y=y_target,
#                     loss_mask=loss_mask,
#                     raw_wind=self.x_raw[:, last, 1:3],
#                     raw_last=self.x_raw[:, last, :])
#
# ============================================================
# CELL 5 — composite_loss: score only the split's nodes
# ============================================================
# Replace the first line of composite_loss:
#
#     final_pred, t_phys, t_res = model_out
#     m = batch.loss_mask
#     if m.sum() < 1:
#         return (torch.zeros((), device=batch.x.device, requires_grad=True),
#                 torch.zeros((), device=batch.x.device))
#     mse_loss = criterion(final_pred[m], batch.y[m])
#
# The push-pull, sparsity and physics terms stay over ALL nodes: they
# are priors on the field, not supervision, and applying them only to
# training nodes would make the residual field discontinuous at block
# boundaries.
#
# Everywhere RMSE is computed in degrees, apply the same mask:
#     err = (out[0][batch.loss_mask] - batch.y[batch.loss_mask]) * temp_std
#
# CELL 9 — the arms and figures are unchanged, but evaluation must use
# batch.loss_mask so test metrics cover held-out BLOCKS only.

## 7 · Feature masks for the ablation arms

In [ ]:
# ============================================================
# 8. FEATURE-MASK HELPERS   (MODIS night LST)
# ------------------------------------------------------------
# Removes CAPABILITIES (inputs, components), not loss weights.
#
# Feature index map:
#    0  t2m         \
#    1  u10          |
#    2  v10          |  ATMOSPHERIC -> physics head
#    3  ssr          |
#    4  str          |
#    5  slhf        /
#    6  NDVI        \
#    7  NDBI         |
#    8  bld_height   |  MORPHOLOGICAL -> residual head
#    9  bld_cover    |
#   10  no2         /
#
# t2m is now maskable. In the ERA5 version index 0 was the target's
# own history and was never masked; here it is an ordinary
# atmospheric predictor, so NoAtmos removes it along with the rest.
#
# MASKING APPLIES TO MODEL INPUT ONLY, and that separation is
# structural rather than a convention: the push-pull term reads
# batch.raw_last (cell 5), a separate unnormalised tensor the mask
# never touches. Dropping morphological INPUTS therefore cannot
# silently disable the morphological CONSTRAINT.
#
# The consequence is intentional and worth stating in the paper: in a
# morphology-dropped arm the model must satisfy a prior defined on
# features it cannot see.
# ============================================================

ATMOS_IDX = [0, 1, 2, 3, 4, 5]
MORPH_IDX = [6, 7, 8, 9, 10]


def make_feature_mask(drop_idx):
    """Multiplicative mask over the feature vector. Zeroing a z-scored
    feature sets it to its training mean -- 'no information', not 'an
    extreme value'. Width comes from FEATURE_COLS so the mask cannot
    silently mismatch the data."""
    n = len(FEATURE_COLS)
    bad = [i for i in drop_idx if not 0 <= i < n]
    if bad:
        raise IndexError(f"feature indices {bad} out of range for {n} features")
    m = torch.ones(n, device=device)
    for i in drop_idx:
        m[i] = 0.0
    return m.view(1, 1, n)


def train_masked(model, train_loader, val_loader, epochs, lr, alpha, gamma,
                 weight_decay, clip, tag, temp_std, feat_mask,
                 single_step=False, patience=40):
    """As train_model, but the model sees x * feat_mask.

    single_step=True collapses the window to its last step, removing
    the LSTM's temporal context -- the ablation that asks whether
    thermal inertia from preceding nights contributes anything.
    """
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5,
                                                 patience=8)
    crit = nn.HuberLoss(delta=1.5)
    best_rmse, best_state, best_epoch, since = float('inf'), None, -1, 0
    t0 = time.time()

    def prep(x):
        x = x * feat_mask
        if single_step:
            x = x[:, -1:, :].repeat(1, x.shape[1], 1)
        return x

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(prep(batch.x), batch.edge_index)
            loss, _ = composite_loss(out, batch, crit, alpha, gamma, temp_std)
            loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
            opt.step()

        model.eval()
        va_loss, sq, n = 0.0, 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(prep(batch.x), batch.edge_index)
                loss, _ = composite_loss(out, batch, crit, alpha, gamma, temp_std)
                va_loss += loss.item()
                # RMSE over VALIDATION BLOCKS only; unmasked, checkpoint
                # selection would be driven partly by training nodes.
                msk = batch.loss_mask
                if msk.sum() < 1:
                    continue
                e = (out[0][msk] - batch.y[msk]) * temp_std
                sq += e.pow(2).sum().item(); n += e.numel()
        va_loss /= max(len(val_loader), 1)
        va_rmse = (sq / max(n, 1)) ** 0.5
        sched.step(va_loss)

        if va_rmse < best_rmse - 1e-4:
            best_rmse, best_epoch, since = va_rmse, epoch + 1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"    [{tag}] {(time.time()-t0)/60:.1f} min | best ep {best_epoch} "
          f"(val RMSE {best_rmse:.3f}°C)")
    return model, prep


def evaluate_masked(model, loader, prep, temp_mean, temp_std, stats_out=None):
    """Returns (rmse_celsius, rBldg). rBldg = corr(t_res, bld_cover).

    bld_cover is read from batch.raw_last, NOT prep(batch.x): in a
    morphology-dropped arm the masked input is zero everywhere, which
    would make rBldg a correlation against a constant rather than a
    measurement. Correlation is unaffected by z-scoring either way.

    rBldg remains a compliance metric, not evidence of discovery. On
    the earlier data it tracked gamma monotonically -- -0.36 at
    gamma=0 rising to +0.70 at gamma=0.1 -- so it measures how hard
    the prior pushed. Report it beside the gamma it was trained at,
    or not at all.
    """
    model.eval()
    P, T, R, B = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            msk = batch.loss_mask
            if msk.sum() < 1:
                continue
            fp, _, tr = model(prep(batch.x), batch.edge_index)
            P.append((fp[msk] * temp_std + temp_mean).cpu().numpy())
            T.append((batch.y[msk] * temp_std + temp_mean).cpu().numpy())
            R.append(tr[msk].cpu().numpy())
            B.append(batch.raw_last[msk, 9].cpu().numpy())    # raw bld_cover
    preds, targ = np.concatenate(P), np.concatenate(T)
    res, bld = np.concatenate(R), np.concatenate(B)

    rmse = float(np.sqrt(mean_squared_error(targ, preds)))
    degenerate = res.std() < 1e-12
    r = float('nan') if degenerate else float(np.corrcoef(res, bld)[0, 1])

    if stats_out is not None:
        stats_out['rmse'] = rmse
        stats_out['r2'] = 1 - float(((targ - preds) ** 2).sum()
                                    / ((targ - targ.mean()) ** 2).sum())
        stats_out['rBldg_pearson'] = r
        stats_out['rBldg_spearman'] = (
            float('nan') if degenerate
            else float(stats.spearmanr(res, bld).statistic))
        stats_out['res_std_c'] = float(res.std() * temp_std)
    return rmse, r

## 8 · Main sweep

Writes `modis_night_results.csv` **unconditionally**. Back that file up before re-running: the morphology ablation in the letter depends on it.

In [ ]:
# ============================================================
# 9. EVALUATION SWEEP + CHECKPOINTS   (MODIS night LST)
# ------------------------------------------------------------
# WHAT THE VALIDATION IS NOW, AND WHAT IT IS NOT.
#
# PRIMARY: held-out night LST. The temporal split reserves the last
# ~25% of night timesteps. The model never sees those nights. This is
# the honest headline number.
#
# READ SPATIAL SKILL, NOT JUST RMSE. Overall R2 on this task is
# dominated by night-to-night variation shared by every node; a model
# that predicts the city-wide level each night and nothing spatial
# still scores well. spatial_skill() splits each timestep into field
# mean plus spatial anomaly and correlates the anomalies. That is the
# number that says whether the graph is contributing.
#
# SECONDARY: Landsat. Kept, but its status has changed and the change
# must be stated. Landsat is a DAYTIME 10:30 observation while the
# model predicts NIGHT LST -- different phenomena, not merely
# different sensors. Daytime surface temperature in these cities is
# weak or inverted with respect to built-up density (corr(bld_cover,
# Landsat LST) ran -0.04 to +0.50), which is exactly why the target
# moved to night. So a modest correlation here is EXPECTED and is not
# evidence of model failure. It is a cross-sensor sanity check on
# spatial pattern, nothing stronger.
#
# ARMS. Full, NoAtmos, NoMorph and NoGraph. NoGraph is a different
# model class, not a feature mask -- same inputs and depth, no message
# passing. It is a first-class arm because on the Landsat version of
# this task ordinary least squares on five features reached R2
# 0.60-0.75 and a random forest beat the graph on cross-city
# transfer. If NoGraph ties Full, the graph is not earning its place.
# ============================================================

ARMS = [
    # (name, model_class, drop_idx)
    ('Full',     'PhyRes',  []),
    ('NoAtmos',  'PhyRes',  ATMOS_IDX),
    ('NoMorph',  'PhyRes',  MORPH_IDX),
    ('NoGraph',  'NoGraph', []),
]
ALPHA = 1.0     # UNVERIFIED on this target -- re-sweep before publishing
GAMMA = 0.03    # UNVERIFIED -- target scale and mask fractions both changed
LR = 8e-4


class MaskedModel(nn.Module):
    """Applies the feature mask inside forward() so every figure
    function works unchanged."""
    def __init__(self, model, prep):
        super().__init__()
        self.model, self.prep = model, prep
    def forward(self, x_seq, edge_index):
        return self.model(self.prep(x_seq), edge_index)
    def get_embeddings(self, x_seq, edge_index):
        return self.model.get_embeddings(self.prep(x_seq), edge_index)


def build_model(kind, n_feat, hp):
    if kind == 'PhyRes':
        return PhyRes(n_feat, hp['spatial_dim'], hp['lstm_hidden'],
                      hp['dropout']).to(device)
    if kind == 'NoGraph':
        return NoGraphModel(n_feat, hp['spatial_dim'], hp['lstm_hidden'],
                            hp['dropout']).to(device)
    raise ValueError(kind)


def load_landsat(cfg, node_ids):
    """Per-node daytime Landsat LST, aligned to the model's node set.

    Positional alignment is not safe: prepare() drops nodes that fail
    the coverage thresholds, so the CSV and the model's node order
    differ. Indexed by id instead.
    """
    df = pd.read_csv(cfg['csv'])
    if 'lst_landsat' not in df.columns:
        return None
    df[ID_COL] = _align_ids(df[ID_COL])
    s = df.groupby(ID_COL)['lst_landsat'].mean()
    return s.reindex(node_ids).values.astype(np.float32)


def mean_residual_map(model, loader):
    """Time-averaged residual per node -- the persistent field."""
    model.eval()
    acc, n = None, 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, _, t_res = model(batch.x, batch.edge_index)
            v = t_res.cpu().numpy()
            acc = v if acc is None else acc + v
            n += 1
    return acc / n


def mean_observed_map(loader):
    """Time-averaged OBSERVED night LST per node, normalised units.

    This is the pattern the residual map should resemble, and it is a
    more appropriate comparison than daytime Landsat.
    """
    acc, n = None, 0
    for batch in loader:
        v = batch.y.cpu().numpy()
        acc = v if acc is None else acc + v
        n += 1
    return acc / n


# ------------------------------------------------------------
print("=" * 72)
print(f"MODIS NIGHT LST — alpha={ALPHA}, gamma={GAMMA}, "
      f"window={HP['window_size']}, horizon={HP['horizon']}")
print("=" * 72)

rows = []
for cfg in CONFIGS:
    city, month = cfg['city'], cfg['month']
    out_dir = f"{OUT_ROOT}/{city}"
    ckpt_dir = f"{out_dir}/{month}_Models"
    os.makedirs(ckpt_dir, exist_ok=True)

    print(f"\n### {city} — {month}")
    # was prepare(CONFIGS[0], HP): every config trained on Bengaluru
    # April, which is why the arms came back identical across cities.
    D = prepare(cfg, HP)
    L = D['pruned']
    node_ids = D['node_ids']
    n_feat = len(FEATURE_COLS)

    report_mask_coverage(L['train'], f"{city} {month}", D['temp_std'])

    landsat = load_landsat(cfg, node_ids)
    if landsat is not None:
        obs_night = mean_observed_map(L['test'])
        ok = np.isfinite(landsat) & np.isfinite(obs_night)
        r_dn = float(np.corrcoef(landsat[ok], obs_night[ok])[0, 1])
        print(f"    day/night agreement in the DATA: "
              f"corr(Landsat day LST, mean night LST) = {r_dn:+.3f}")
        print(f"    This bounds what any model can score against Landsat.")

    for tag, kind, drop in ARMS:
        set_seed(HP['seed'])
        fmask = make_feature_mask(drop)
        raw = build_model(kind, n_feat, HP)
        raw, prep = train_masked(raw, L['train'], L['val'], HP['epochs'],
                                 lr=LR, alpha=ALPHA, gamma=GAMMA,
                                 weight_decay=1e-4, clip=0.5, tag=tag,
                                 temp_std=D['temp_std'], feat_mask=fmask)

        torch.save({'state_dict': raw.state_dict(), 'feature_mask': fmask.cpu(),
                    'kind': kind, 'arm': tag, 'alpha': ALPHA, 'gamma': GAMMA,
                    'cover_t': COVER_T, 'ndvi_t': NDVI_T,
                    'push_c': PUSH_MARGIN_C, 'pull_c': PULL_MARGIN_C,
                    'city': city, 'month': month,
                    'temp_mean': D['temp_mean'], 'temp_std': D['temp_std']},
                   f"{ckpt_dir}/{month}_{tag}.pt")

        m = MaskedModel(raw, prep).to(device)
        p = f"{out_dir}/{month}_{tag}"

        if tag == ARMS[0][0]:
            loss_breakdown(m, L['val'], ALPHA, GAMMA, D['temp_std'],
                           f"{city} {month} {tag}")

        metrics, _, _, _ = diagnostic_eval(
            m, L['test'], D['temp_mean'], D['temp_std'],
            f"{p}_Scatter+Hist.png", f"{city} {month} — {tag}")
        sk = spatial_skill(m, L['test'], D['temp_mean'], D['temp_std'], tag)
        corr = residual_correlations(m, L['test'], f"{p}_Corr.png",
                                     f"{city} {month} — {tag}")
        collapse = collapse_check(m, L['test'], f"{p}_Collapse.png",
                                  f"{city} {month} — {tag}")
        city_residual_map(m, L['test'], D['gdf'], D['temp_std'],
                          f"{p}_City.png",
                          f"{city} {month} — {tag} nocturnal UHI increment")

        # cross-sensor check (day vs night -- see header)
        r_lst = float('nan')
        if landsat is not None:
            res_map = mean_residual_map(m, L['test'])
            ok = np.isfinite(landsat) & np.isfinite(res_map)
            if res_map.std() > 1e-12 and ok.sum() > 50:
                r_lst = float(np.corrcoef(res_map[ok], landsat[ok])[0, 1])

        print(f"    {tag:<9} RMSE {metrics['rmse']:.3f}°C | R² {metrics['r2']:+.3f}"
              f" | spatial r {sk['spatial_r_mean']:+.3f}"
              f" | collapse {collapse:.2f}%"
              f" | rBldg {corr['Building coverage']:+.3f}"
              f" | r(t_res, Landsat) {r_lst:+.3f}")

        rows.append(dict(city=city, month=month, arm=tag, kind=kind,
                         rmse=metrics['rmse'], mae=metrics['mae'],
                         r2=metrics['r2'], res_std_c=metrics['res_std'],
                         collapse=collapse,
                         rBldg=corr['Building coverage'],
                         rVeg=corr['Vegetation (NDVI)'],
                         r_res_landsat=r_lst,
                         gap_frac=D['gap_frac'],
                         spatial_sd=D['spatial_sd'],
                         n_nodes=D['n_nodes'], n_time=D['n_time'],
                         alpha=ALPHA, gamma=GAMMA, **sk))
        plot_model_diagnostics(m, D, CONFIGS[0], 'Full')
        del raw, m
        if device.type == 'cuda':
            torch.cuda.empty_cache()

v = pd.DataFrame(rows)
v.to_csv(f"{OUT_ROOT}/modis_night_results.csv", index=False)

print("\n" + "=" * 72)
print(v[['city', 'month', 'arm', 'rmse', 'r2', 'spatial_r_mean',
         'spatial_spread_ratio', 'rBldg', 'r_res_landsat']]
      .round(3).to_string(index=False))

print("\n" + "=" * 72)
print("READING THIS")
print("=" * 72)
print("""
1. spatial_r_mean is the number that matters. RMSE and overall R2 are
   dominated by night-to-night variation every node shares; a model
   with no spatial skill at all can still post a good RMSE.

2. Compare Full against NoGraph. Same inputs, same depth, no message
   passing. If they tie, the graph is not contributing and the paper
   must say so -- on the Landsat version, a random forest on five
   features matched or beat the graph.

3. Compare Full against NoMorph. If dropping morphology costs little,
   the urban form signal is not being used and the residual head is
   decorative.

4. spatial_spread_ratio well below 1 means predictions are flatter
   than reality -- the model hedging toward the field mean, which is
   what squared-error minimisation rewards when the pattern cannot be
   resolved.

5. rBldg is a compliance metric, not a discovery. It tracked gamma
   monotonically on the earlier data. Quote it beside its gamma or
   leave it out.

6. r(t_res, Landsat) is day-versus-night. The printed
   corr(Landsat, observed night LST) above bounds it: no model can
   exceed the agreement present in the data itself.
""")

## 9 · Diagnostic plots

In [ ]:
# ============================================================
# 10. DIAGNOSTIC PLOTS
# ------------------------------------------------------------
# Two sets.
#
# plot_data_diagnostics(cfg, D) runs BEFORE or independently of any
# model. It shows what the data actually looks like: where cloud
# masking left holes, how thin the splits are, whether the target has
# spatial structure, and whether the push-pull thresholds select a
# sensible fraction of nodes. Most failures visible here cannot be
# fixed by the model.
#
# plot_model_diagnostics(model, D, ...) runs after training and shows
# where the model breaks: whether it tracks the night-to-night level
# but not the spatial pattern, whether the two heads are doing
# separate jobs or cancelling, and which parts of the city it gets
# wrong.
#
# Cell 3 sets a headless backend, so figures are saved and then
# displayed rather than shown directly.
# ============================================================
from IPython.display import Image, display


def _show(fig, path):
    fig.savefig(path, dpi=110, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=path))


def plot_data_diagnostics(cfg, D, save_dir=None):
    """What the data looks like before the model touches it."""
    save_dir = save_dir or f"{OUT_ROOT}/{cfg['city']}"
    os.makedirs(save_dir, exist_ok=True)
    tag = f"{cfg['city']} {cfg['month']}"

    # re-read to see pre-interpolation validity
    df = pd.read_csv(cfg['csv'])
    df[ID_COL] = _align_ids(df[ID_COL])
    if np.issubdtype(df[TIME_COL].dtype, np.number):
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], unit='ms')
    else:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    if NIGHT_ONLY:
        df = df[df['pass'].str.contains('night', na=False)]
    df = df[df[ID_COL].isin(D['node_ids'])]

    fig = plt.figure(figsize=(16, 11))
    gs = fig.add_gridspec(3, 3, hspace=0.38, wspace=0.28)

    # --- 1. validity: which node x time cells were actually observed
    ax = fig.add_subplot(gs[0, :2])
    piv = df.pivot_table(index=ID_COL, columns=TIME_COL, values='lst_modis')
    ax.imshow(piv.notna().values, aspect='auto', cmap='Greys_r',
              interpolation='nearest')
    ax.set_title(f"{tag} — observed (white) vs cloud-masked (black)\n"
                 f"{100*piv.notna().values.mean():.0f}% observed before "
                 f"interpolation", fontsize=10)
    ax.set_xlabel("night pass index"); ax.set_ylabel("node")

    # --- 2. gaps per node
    ax = fig.add_subplot(gs[0, 2])
    ax.hist(piv.notna().sum(axis=1).values, bins=25, color='steelblue')
    ax.axvline(HP['window_size'], color='r', ls='--',
               label=f"window={HP['window_size']}")
    ax.set_title("valid passes per node", fontsize=10)
    ax.set_xlabel("count"); ax.legend(fontsize=8)

    # --- 3. field-mean timeline with split boundaries
    ax = fig.add_subplot(gs[1, :])
    fm = df.groupby(TIME_COL)['lst_modis'].mean()
    sd = df.groupby(TIME_COL)['lst_modis'].std()
    ax.plot(range(len(fm)), fm.values, 'o-', color='k', ms=4,
            label='city mean')
    ax.fill_between(range(len(fm)), fm.values - sd.values,
                    fm.values + sd.values, alpha=0.25, color='steelblue',
                    label='+/- spatial sd')
    tr_end = int(D['n_time'] * HP['train_frac'])
    va_end = tr_end + max(1, int(D['n_time'] * HP['val_frac']))
    for x, lab, c in ((tr_end, 'train|val', 'green'), (va_end, 'val|test', 'red')):
        ax.axvline(x - 0.5, color=c, ls='--', lw=1.5, label=lab)
    ax.set_title(f"{tag} — night LST over time. The shaded band is the "
                 f"spatial spread: that is what the graph has to explain.",
                 fontsize=10)
    ax.set_xlabel("night pass index"); ax.set_ylabel("°C")
    ax.legend(fontsize=8, ncol=4)

    # --- 4. observed mean night LST map
    ax = fig.add_subplot(gs[2, 0])
    g = D['gdf'].copy()
    obs = df.groupby(ID_COL)['lst_modis'].mean().reindex(D['node_ids']).values
    g['obs'] = obs
    g.plot(column='obs', cmap='inferno', ax=ax, legend=True,
           legend_kwds={'shrink': 0.7})
    ax.set_title("mean observed night LST (°C)", fontsize=10); ax.set_axis_off()

    # --- 5. bld_cover with the push-pull threshold marked
    ax = fig.add_subplot(gs[2, 1])
    cov = df.groupby(ID_COL)['bld_cover'].mean().reindex(D['node_ids']).values
    ax.hist(cov, bins=30, color='sienna')
    ax.axvline(COVER_T, color='r', ls='--',
               label=f"COVER_T={COVER_T}\nselects {100*(cov>COVER_T).mean():.1f}%")
    ax.set_title("building coverage", fontsize=10)
    ax.set_xlabel("fraction"); ax.legend(fontsize=8)

    # --- 6. the relationship the whole design rests on
    ax = fig.add_subplot(gs[2, 2])
    ok = np.isfinite(cov) & np.isfinite(obs)
    ax.scatter(cov[ok], obs[ok], s=6, alpha=0.4, color='darkred')
    if ok.sum() > 2:
        z = np.polyfit(cov[ok], obs[ok], 1)
        xs = np.linspace(cov[ok].min(), cov[ok].max(), 40)
        ax.plot(xs, np.polyval(z, xs), 'k--')
        r = np.corrcoef(cov[ok], obs[ok])[0, 1]
        ax.set_title(f"cover vs night LST   r={r:+.3f}", fontsize=10)
    ax.set_xlabel("bld_cover"); ax.set_ylabel("night LST (°C)")

    _show(fig, f"{save_dir}/{cfg['month']}_DataDiagnostics.png")


def plot_model_diagnostics(model, D, cfg, tag, save_dir=None):
    """Where the model succeeds and where it breaks."""
    save_dir = save_dir or f"{OUT_ROOT}/{cfg['city']}"
    os.makedirs(save_dir, exist_ok=True)
    tm, ts = D['temp_mean'], D['temp_std']

    model.eval()
    P, Y, PH, TR, COV = [], [], [], [], []
    with torch.no_grad():
        for batch in D['pruned']['test']:
            batch = batch.to(device)
            fp, tp, tr = model(batch.x, batch.edge_index)
            P.append(fp.cpu().numpy()); Y.append(batch.y.cpu().numpy())
            PH.append(tp.cpu().numpy()); TR.append(tr.cpu().numpy())
            COV.append(batch.raw_last[:, 9].cpu().numpy())
    P, Y = np.stack(P), np.stack(Y)          # [windows, nodes]
    PH, TR = np.stack(PH), np.stack(TR)
    cov = COV[0]

    Pc, Yc = P * ts + tm, Y * ts + tm
    Pa, Ya = P - P.mean(1, keepdims=True), Y - Y.mean(1, keepdims=True)

    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.38, wspace=0.30)

    # --- 1. does it track the night-to-night level?
    ax = fig.add_subplot(gs[0, :2])
    ax.plot(Yc.mean(1), 'o-', color='k', ms=5, label='observed city mean')
    ax.plot(Pc.mean(1), 's--', color='crimson', ms=5, label='predicted')
    ax.set_title(f"{tag} — city-mean night LST across test windows. Easy to "
                 f"get right; it is not the interesting part.", fontsize=10)
    ax.set_xlabel("test window"); ax.set_ylabel("°C"); ax.legend(fontsize=8)

    # --- 2. spatial skill per timestep
    ax = fig.add_subplot(gs[0, 2])
    rs = [np.corrcoef(Ya[t], Pa[t])[0, 1] for t in range(Ya.shape[0])
          if Ya[t].std() > 1e-9 and Pa[t].std() > 1e-9]
    ax.plot(rs, 'o-', color='teal', ms=4)
    ax.axhline(0, color='k', lw=0.8)
    ax.axhline(np.mean(rs), color='r', ls='--',
               label=f"mean {np.mean(rs):+.3f}")
    ax.set_ylim(-1, 1)
    ax.set_title("spatial pattern correlation\n(THE number that matters)",
                 fontsize=10)
    ax.set_xlabel("test window"); ax.legend(fontsize=8)

    # --- 3-5. observed / predicted / error maps, shared scale
    g = D['gdf'].copy()
    g['obs'] = Yc.mean(0); g['pred'] = Pc.mean(0)
    g['err'] = g['pred'] - g['obs']
    vmin, vmax = min(g['obs'].min(), g['pred'].min()), max(g['obs'].max(), g['pred'].max())
    for i, (col, cmap, ttl, kw) in enumerate([
            ('obs', 'inferno', 'observed mean', dict(vmin=vmin, vmax=vmax)),
            ('pred', 'inferno', 'predicted mean', dict(vmin=vmin, vmax=vmax)),
            ('err', 'coolwarm', 'error (pred - obs)',
             dict(vmin=-float(np.abs(g['err']).max()),
                  vmax=float(np.abs(g['err']).max())))]):
        ax = fig.add_subplot(gs[1, i])
        g.plot(column=col, cmap=cmap, ax=ax, legend=True,
               legend_kwds={'shrink': 0.6}, **kw)
        ax.set_title(f"{ttl} (°C)", fontsize=10); ax.set_axis_off()

    # --- 6. head decomposition maps
    ax = fig.add_subplot(gs[2, 0])
    g['phys'] = PH.mean(0) * ts
    g.plot(column='phys', cmap='viridis', ax=ax, legend=True,
           legend_kwds={'shrink': 0.6})
    ax.set_title(f"t_phys (sd {PH.std()*ts:.2f}°C)\nflat = no spatial role",
                 fontsize=10)
    ax.set_axis_off()

    ax = fig.add_subplot(gs[2, 1])
    g['res'] = TR.mean(0) * ts
    lim = float(np.abs(g['res']).max())
    g.plot(column='res', cmap='coolwarm', ax=ax, legend=True,
           vmin=-lim, vmax=lim, legend_kwds={'shrink': 0.6})
    ax.set_title(f"t_res (sd {TR.std()*ts:.2f}°C)\nthe UHI increment",
                 fontsize=10)
    ax.set_axis_off()

    # --- 7. does the residual track building coverage?
    ax = fig.add_subplot(gs[2, 2])
    rm = TR.mean(0) * ts
    ax.scatter(cov, rm, s=6, alpha=0.4, color='darkgreen')
    ax.axvline(COVER_T, color='r', ls='--', lw=1,
               label=f"push fires above {COVER_T}")
    ax.axhline(0, color='k', lw=0.8)
    if np.std(cov) > 0 and np.std(rm) > 0:
        r = np.corrcoef(cov, rm)[0, 1]
        ax.set_title(f"t_res vs bld_cover   r={r:+.3f}\n"
                     f"(compliance with the prior, not discovery)",
                     fontsize=10)
    ax.set_xlabel("bld_cover"); ax.set_ylabel("t_res (°C)")
    ax.legend(fontsize=8)

    _show(fig, f"{save_dir}/{cfg['month']}_{tag}_ModelDiagnostics.png")

    # --- printed summary of the diagnoses the plots support ------
    print(f"\n  [{tag}] diagnosis")
    print(f"    spatial r          {np.mean(rs):+.3f}"
          + ("   <-- little spatial skill" if np.mean(rs) < 0.2 else ""))
    print(f"    spread ratio       {Pa.std()/Ya.std():.2f}"
          + ("   <-- predictions flatter than reality"
             if Pa.std()/Ya.std() < 0.5 else ""))
    print(f"    t_phys sd          {PH.std()*ts:.3f} °C"
          + ("   <-- physics head is a constant"
             if PH.std()*ts < 0.05 else ""))
    print(f"    t_res  sd          {TR.std()*ts:.3f} °C"
          + ("   <-- residual head is a constant"
             if TR.std()*ts < 0.05 else ""))
    tra = TR - TR.mean(1, keepdims=True)
    share = tra.std() / Pa.std() if Pa.std() > 0 else np.nan
    print(f"    residual supplies  {share:.0%} of predicted spatial spread"
          + ("   <-- heads are cancelling, not decomposing"
             if share > 1.5 else ""))


# ------------------------------------------------------------
# USAGE
#
#   D = prepare(CONFIGS[0], HP)
#   plot_data_diagnostics(CONFIGS[0], D)          # before training
#
#   # after training an arm (m is the MaskedModel from cell 9):
#   plot_model_diagnostics(m, D, CONFIGS[0], 'Full')
#
# WHAT TO LOOK FOR
#   panel 1  large black regions -> heavy cloud masking, and the
#            interpolated fraction is doing more work than you think
#   panel 3  if the shaded spatial band is thin relative to the line's
#            up-and-down movement, the target is mostly temporal and
#            the graph has little to explain
#   panel 6  a flat t_phys map means the physics head collapsed to a
#            constant, which is what happened in three of four
#            NoAtmos arms on the ERA5 data -- there is then no
#            decomposition, only a residual doing all the work
#   panel 7  t_res rising with bld_cover shows the prior was
#            satisfied. It does NOT show the model discovered the
#            relationship; gamma controls it directly.
# ------------------------------------------------------------

## 10 · Tree baselines

Random forest and gradient boosting on the same blocked splits, with and without spatial-lag features. Hyperparameters are selected on the validation blocks, not on a random row split.

In [ ]:
"""
GRSL supplementary experiments: tree baselines, model cost, map export.

Covers three reviewer-facing gaps in one pass:
  (1) non-neural baselines (RF / HistGradientBoosting), with and without
      spatial-lag features, on the SAME spatially blocked splits
  (2) learnable-parameter counts and wall-clock for every architecture
  (3) observed / predicted / residual arrays for the qualitative map figure

Nothing here retrains a GNN. Item (2) times the existing architectures on a
short fixed budget purely to report cost; it does not produce new accuracy
numbers.

USAGE
  Fill in SECTION 0 (the adapter). Everything below consumes plain numpy.
  Run in Colab after your notebook has defined prepare() / CONFIGS.

OUTPUTS (to OUT_DIR)
  tree_baselines.csv     per config x model: RMSE, spatial r, R2
  model_cost.csv         per architecture: params, train s/epoch, inference ms
  map_export_<cfg>.csv   node_id, x, y, block, obs, pred, resid, n_valid
"""

import json
import os
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# SECTION 0 — ADAPTER.  This is the only part you edit.
# =============================================================================
# Wired to the notebook's prepare() / PhyResDataset / PhyRes / NoGraphModel.
# Run cells 2, 3, 4, 5, 7, 8 first so CONFIGS, HP, prepare, PhyRes,
# NoGraphModel, knn_graph and device are defined.
# =============================================================================

import numpy as np
import torch

OUT_DIR = "/content/drive/MyDrive/UHI/PhyRes_MODIS/grsl_supp"
MAP_CONFIG = "Delhi_April"   # matches the label built in load_configs()
SEEDS = (0,)
N_LAG_NEIGHBOURS = 8         # k used by the MLP+lag model
WINDOW_MODE = "flatten"

FEATURE_NAMES = [
    "t2m", "u10", "v10", "ssr", "str", "slhf",
    "NDVI", "NDBI", "bld_height", "bld_cover", "no2",
]
ATMOS_SLICE = slice(0, 6)
URBAN_SLICE = slice(6, 11)

# Morphological channels that get a neighbourhood mean appended in the
# MLP+lag arm. Atmospherics are excluded: ERA5 is far coarser than the
# 1 km node spacing, so its neighbourhood mean is nearly a copy of itself
# (measured: corr(own, lag) = 0.996 for no2, which is ~7 km native).
LAG_CHANNELS = [6, 7, 8, 9, 10]


def load_configs():
    """Labels for the six city-season configurations."""
    return [f"{c['city']}_{c['month']}" for c in CONFIGS]


def _cfg_by_label(label):
    for c in CONFIGS:
        if f"{c['city']}_{c['month']}" == label:
            return c
    raise KeyError(f"{label!r} not in {load_configs()}")


def load_config(cfg):
    """One config as plain numpy, in the contract the script expects.

    `cfg` may be a label from load_configs() or a CONFIGS dict.

    Note on shapes: prepare() hands back full [N, T_all, F] tensors plus
    a window_size, and the dataset slices windows on the fly. Here the
    windows are materialised into (N, T, W, F), so memory grows by a
    factor of W. At 830 nodes x 104 windows x 5 steps x 11 features that
    is ~19 MB in float32 -- fine, but worth knowing before raising
    window_size.
    """
    if isinstance(cfg, str):
        cfg = _cfg_by_label(cfg)

    D = prepare(cfg, HP)
    ds = D['pruned']['train'].dataset       # all three splits share the
                                            # same tensors and differ only
                                            # by node_mask
    W = ds.window_size
    H = ds.horizon
    n_win = ds.num_samples

    x_norm = ds.x_norm.detach().cpu().numpy()          # [N, T_all, F]
    y_norm = ds.y.detach().cpu().numpy()               # [N, T_all]
    observed = ds.observed.detach().cpu().numpy()      # [N, T_all] bool

    # window i covers steps [i, i+W); its target is at i+W-1+H
    starts = np.arange(n_win)
    tgt_idx = starts + W - 1 + H

    # (n_win, N, W, F) -> (N, n_win, W, F)
    X = np.stack([x_norm[:, i:i + W, :] for i in starts], axis=0)
    X = np.transpose(X, (1, 0, 2, 3)).astype(np.float32)

    # de-normalise the target so RMSE comes out in degrees Celsius
    y = (y_norm[:, tgt_idx] * D['temp_std'] + D['temp_mean']).astype(np.float32)
    valid = observed[:, tgt_idx]

    # split label per node, from the blocked masks
    split = np.empty(D['n_nodes'], dtype=object)
    for name in ('train', 'val', 'test'):
        split[D['masks'][name].detach().cpu().numpy()] = name
    if (split == None).any():                                    # noqa: E711
        raise ValueError("some nodes fall in no split -- check prepare()")

    out = dict(
        X=X,
        y=y,
        valid=valid.astype(bool),
        pos=ds.pos.detach().cpu().numpy().astype(np.float32),
        block=np.asarray(D['block'], dtype=int),
        split=split.astype(str),
        node_id=np.asarray(D['node_ids']),
    )

    N, T = out['y'].shape
    assert out['X'].shape == (N, T, W, len(FEATURE_NAMES)), out['X'].shape
    assert out['valid'].shape == (N, T)
    assert out['pos'].shape == (N, 2)
    assert out['block'].shape == (N,) and out['split'].shape == (N,)
    assert len(out['node_id']) == N
    print(f"  {cfg['city']} {cfg['month']}: {N} nodes, {T} windows, "
          f"W={W}, {100*out['valid'].mean():.1f}% genuinely observed")
    return out


def build_architectures(n_features_flat, n_features_graph):
    """The same classes and hyperparameters the main sweep used, so the
    reported parameter counts are the real ones.

    MLP+lag takes n_features_graph + len(LAG_CHANNELS), not 2x: only the
    morphological channels get a neighbourhood mean appended.
    """
    kw = dict(spatial_dim=HP['spatial_dim'], lstm_hidden=HP['lstm_hidden'],
              dropout=HP['dropout'])
    n_lag = n_features_graph + len(LAG_CHANNELS)
    return {
        "GAT+skip k=8": (PhyRes(n_features_graph, **kw), True),
        "GAT+skip k=0": (PhyRes(n_features_graph, **kw), True),
        "MLP":          (NoGraphModel(n_features_graph, **kw), False),
        "MLP+lag":      (NoGraphModel(n_lag, **kw), False),
    }


# ---------------------------------------------------------------------------
# Quick self-check. Run this before the rest of the script.
# ---------------------------------------------------------------------------
def _check(label=None):
    labels = load_configs()
    print("configs:", labels)
    d = load_config(label or labels[0])
    N, T = d['y'].shape
    for name in ('train', 'val', 'test'):
        m = d['split'] == name
        print(f"  {name:<6}{m.sum():>5} nodes, "
              f"{d['valid'][m].mean()*100:5.1f}% observed")
    print(f"  target {np.nanmin(d['y']):.1f} to {np.nanmax(d['y']):.1f} °C "
          f"(should look like night LST, roughly 15-35 °C)")
    print(f"  pos span {np.ptp(d['pos'][:,0]):,.0f} x "
          f"{np.ptp(d['pos'][:,1]):,.0f} m")
    print(f"  blocks {len(np.unique(d['block']))} distinct")
    if len(build_architectures(len(FEATURE_NAMES), len(FEATURE_NAMES))):
        for k, (m, g) in build_architectures(len(FEATURE_NAMES),
                                             len(FEATURE_NAMES)).items():
            print(f"  {k:<14}{sum(p.numel() for p in m.parameters()):>9,} params"
                  f"  graph={g}")
    return d

_check()
SPATIAL_R_MODE = "per_instant"
MIN_NODES_PER_INSTANT = 5
N_LAG_NEIGHBOURS = 8
LAG_CHANNELS = [6, 7, 8, 9, 10]
N_FEATURES_BY_ARCH = {}
# =============================================================================
# SECTION 1 — metrics and feature construction
# =============================================================================

def spatial_pearson(y_true, y_pred, valid, min_nodes=MIN_NODES_PER_INSTANT,
                    return_n=False, mode=SPATIAL_R_MODE):
    """Across-node agreement between predicted and observed fields.

    mode selects the axis convention. These are NOT interchangeable:

      "per_instant"  correlate across nodes within each instant, then
                     average. Measures spatial pattern only.
      "pooled"       one correlation over all (node, instant) pairs.
                     INFLATED here: temporal sd is ~2.3 C against spatial
                     ~0.8 C, so this mostly measures whether the model
                     tracks which night it is.
      "node_anomaly" subtract each node's temporal mean first, then pool.
                     DEFLATED here: the spatial pattern largely IS the
                     per-node mean structure, so this removes the signal
                     and correlates the leftovers.

    Note that subtracting a scalar mean before a Pearson correlation has no
    effect at all -- Pearson centres internally. Only the axis matters.

    Non-finite correlations are dropped rather than averaged; a single NaN
    instant otherwise turns the whole config's mean into NaN. Instants with
    fewer than min_nodes valid nodes are excluded, since a correlation over
    ~10 points has a standard error near 0.3.

    Whichever mode and min_nodes you use must appear in the paper next to
    n_instants, and must match the main sweep or the numbers are not
    comparable.
    """
    if mode == "pooled":
        m = valid
        a, b = y_true[m], y_pred[m]
        ok = np.isfinite(a) & np.isfinite(b)
        a, b = a[ok], b[ok]
        if a.size < min_nodes or a.std() < 1e-8 or b.std() < 1e-8:
            return (np.nan, 0) if return_n else np.nan
        r = float(np.corrcoef(a, b)[0, 1])
        return (r, 1) if return_n else r

    if mode == "node_anomaly":
        n_obs = valid.sum(axis=1, keepdims=True)
        def demean(z):
            s = np.where(valid, z, 0.0).sum(axis=1, keepdims=True)
            return z - np.divide(s, np.maximum(n_obs, 1))
        ya, pa = demean(y_true), demean(y_pred)
        m = valid & (n_obs > 1)
        a, b = ya[m], pa[m]
        ok = np.isfinite(a) & np.isfinite(b)
        a, b = a[ok], b[ok]
        if a.size < min_nodes or a.std() < 1e-8 or b.std() < 1e-8:
            return (np.nan, 0) if return_n else np.nan
        r = float(np.corrcoef(a, b)[0, 1])
        return (r, 1) if return_n else r

    if mode != "per_instant":
        raise ValueError(f"unknown SPATIAL_R_MODE: {mode}")

    rs = []
    for t in range(y_true.shape[1]):
        m = valid[:, t]
        if m.sum() < min_nodes:
            continue
        a, b = y_true[m, t], y_pred[m, t]
        if not (np.isfinite(a).all() and np.isfinite(b).all()):
            continue
        if a.std() < 1e-8 or b.std() < 1e-8:
            continue
        r = np.corrcoef(a, b)[0, 1]
        if np.isfinite(r):
            rs.append(r)
    val = float(np.nanmean(rs)) if rs else np.nan
    return (val, len(rs)) if return_n else val


def reconcile_spatial_r(y_true, y_pred, valid, cell6_value=None):
    """Print every convention on the same predictions.

    Run this on a stored checkpoint's predictions for a config whose main-sweep
    spatial r you already know. Whichever row reproduces that number is the
    convention the sweep used, and is what SPATIAL_R_MODE must be set to before
    any tree-vs-neural comparison means anything.
    """
    print(f"  {'mode':<14}{'min_nodes':>10}{'r':>9}{'n':>6}")
    for mode in ("per_instant", "pooled", "node_anomaly"):
        for mn in (5, 30):
            r, n = spatial_pearson(y_true, y_pred, valid, min_nodes=mn,
                                   return_n=True, mode=mode)
            flag = ""
            if cell6_value is not None and np.isfinite(r):
                flag = "   <-- matches" if abs(r - cell6_value) < 0.005 else ""
            print(f"  {mode:<14}{mn:>10}{r:>9.3f}{n:>6}{flag}")
    if cell6_value is not None:
        print(f"  cell 6 reported: {cell6_value:.3f}")


def masked_scores(y_true, y_pred, valid):
    m = valid
    if m.sum() == 0:
        return dict(rmse=np.nan, mae=np.nan, r2=np.nan,
                    spatial_r=np.nan, n_instants=0)
    err = y_pred[m] - y_true[m]
    ss_res = float((err ** 2).sum())
    ss_tot = float(((y_true[m] - y_true[m].mean()) ** 2).sum())
    sr, n_inst = spatial_pearson(y_true, y_pred, valid, return_n=True)
    return dict(
        rmse=float(np.sqrt((err ** 2).mean())),
        mae=float(np.abs(err).mean()),
        r2=1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
        spatial_r=sr,
        n_instants=n_inst,
    )


def knn_indices(pos, k):
    """k nearest neighbours by Euclidean distance, self excluded. (N, k) int."""
    d2 = ((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    return np.argsort(d2, axis=1)[:, :k]


def add_spatial_lag(X_flat, nbr_idx, W=None, F=None, channels=None):
    """Concatenate neighbour means of selected channels, at every window step.

    Only PREDICTORS are averaged, never the target. Neighbour features are
    known everywhere at inference time, so borrowing them across block
    boundaries is not leakage -- borrowing neighbour targets would be. Say
    this explicitly in the method section.

    X_flat: (N, T, W*F) -> (N, T, W*F + W*len(channels))
    """
    if F is None:
        F = len(FEATURE_NAMES)
    if W is None:
        W = X_flat.shape[-1] // F
    channels = LAG_CHANNELS if channels is None else channels
    N, T, D = X_flat.shape
    assert D == W * F, f"flattened width {D} != W*F = {W}*{F}"
    cube = X_flat.reshape(N, T, W, F)
    lag = cube[nbr_idx][:, :, :, :, channels].mean(axis=1)   # (N,T,W,len(ch))
    return np.concatenate(
        [X_flat, lag.reshape(N, T, W * len(channels))], axis=-1
    ).astype(np.float32)


def build_design_matrix(X, mode=WINDOW_MODE):
    """(N, T, W, F) -> (N, T, D).

    "flatten" gives the tree the whole window (D = W*F), matching the
    information the sequence models see. "last" uses only the final step
    (D = F), which is a weaker but more interpretable baseline. Report
    whichever you use; flatten is the fairer comparison.
    """
    N, T, W, F = X.shape
    if mode == "flatten":
        return X.reshape(N, T, W * F).astype(np.float32)
    if mode == "last":
        return X[:, :, -1, :].astype(np.float32)
    raise ValueError(f"unknown WINDOW_MODE: {mode}")


def stack_rows(D, y, valid, node_mask):
    """Flatten (node, instant) pairs into rows, keeping only valid targets."""
    sel = node_mask[:, None] & valid
    idx = np.argwhere(sel)
    rows = D[idx[:, 0], idx[:, 1], :]
    targ = y[idx[:, 0], idx[:, 1]]
    return rows, targ, idx


# =============================================================================
# SECTION 2 — tree baselines
# =============================================================================

def candidate_models(seed):
    """Small hyperparameter grid per model family.

    Selection happens on the spatially blocked VALIDATION blocks, not on a
    random row split. sklearn's internal early stopping splits randomly over
    rows, and with stride-1 windows one node contributes many near-identical
    rows, so the same node lands on both sides of that split. Using the val
    blocks avoids this and gives a justifiable answer to "how were the
    hyperparameters chosen?".
    """
    from sklearn.ensemble import (HistGradientBoostingRegressor,
                                  RandomForestRegressor)
    return {
        "RandomForest": [
            (f"leaf{leaf}", RandomForestRegressor(
                n_estimators=300, min_samples_leaf=leaf, max_features="sqrt",
                n_jobs=-1, random_state=seed))
            for leaf in (1, 2, 8)
        ],
        "HistGBR": [
            (f"it{it}_lr{lr}", HistGradientBoostingRegressor(
                max_iter=it, learning_rate=lr, max_leaf_nodes=31,
                early_stopping=False, random_state=seed))
            for it, lr in ((200, 0.06), (400, 0.06), (800, 0.03))
        ],
    }


def run_tree_baselines(cfg):
    d = load_config(cfg)
    X, y, valid = d["X"], d["y"], d["valid"]
    pos, split = d["pos"], d["split"]

    tr = split == "train"
    te = split == "test"

    D_plain = build_design_matrix(X)
    nbr = knn_indices(pos, N_LAG_NEIGHBOURS)
    D_lag = add_spatial_lag(D_plain, nbr)

    va = split == "val"

    def scatter(pred_flat, idx):
        pred = np.zeros_like(y, dtype=np.float64)
        pred[idx[:, 0], idx[:, 1]] = pred_flat
        mask = np.zeros_like(valid)
        mask[idx[:, 0], idx[:, 1]] = True
        return pred, mask

    rows = []
    for feat_name, D in (("plain", D_plain), ("+lag", D_lag)):
        Xtr, ytr, _ = stack_rows(D, y, valid, tr)
        Xva, yva, idx_va = stack_rows(D, y, valid, va)
        Xte, yte, idx_te = stack_rows(D, y, valid, te)
        for seed in SEEDS:
            for mname, candidates in candidate_models(seed).items():
                best, best_rmse, best_label = None, np.inf, None
                for label, cand in candidates:
                    cand.fit(Xtr, ytr)
                    rmse_va = float(np.sqrt(((cand.predict(Xva) - yva) ** 2).mean()))
                    if rmse_va < best_rmse:
                        best, best_rmse, best_label = cand, rmse_va, label

                t0 = time.perf_counter()
                best.fit(Xtr, ytr)
                fit_s = time.perf_counter() - t0

                t0 = time.perf_counter()
                pred_flat = best.predict(Xte)
                pred_ms = (time.perf_counter() - t0) * 1000.0

                pred, te_valid = scatter(pred_flat, idx_te)
                s = masked_scores(y, pred, te_valid)
                rows.append(dict(
                    config=cfg, model=f"{mname} {feat_name}", features=feat_name,
                    seed=seed, selected=best_label, val_rmse=round(best_rmse, 4),
                    n_train_rows=len(ytr), n_val_rows=len(yva),
                    n_test_rows=len(yte), n_input_dims=D.shape[-1],
                    fit_seconds=round(fit_s, 2), predict_ms=round(pred_ms, 2),
                    **s,
                ))
                print(f"  {cfg:26s} {mname:14s} {feat_name:6s} seed={seed}  "
                      f"[{best_label:10s}]  RMSE={s['rmse']:.3f}  "
                      f"spatial_r={s['spatial_r']:.3f}  "
                      f"(n_inst={s['n_instants']})")
    return rows


# =============================================================================
# SECTION 3 — parameter counts and wall-clock
# =============================================================================

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def measure_cost(cfg, n_warmup=2, n_timed=5):
    """Time one training step and one forward pass per architecture.

    Reports seconds per training step on a fixed synthetic batch shaped like
    the real data, so the comparison is like-for-like. Not an accuracy run.
    """
    try:
        import torch
    except ImportError:
        print("  torch unavailable; skipping cost accounting")
        return []

    d = load_config(cfg)
    X = d["X"]
    N, T, W, F = X.shape
    n_flat = W * F
    archs = build_architectures(n_flat, F)
    if not archs:
        print("  build_architectures() empty; skipping cost accounting")
        return []

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    nbr = knn_indices(d["pos"], 8)
    ei_full = torch.from_numpy(np.stack([
        nbr.reshape(-1), np.repeat(np.arange(N), 8)]).astype(np.int64)).to(dev)
    ei_empty = torch.zeros((2, 0), dtype=torch.long, device=dev)

    target = torch.randn(N, device=dev)

    rows = []
    for name, (model, needs_graph) in archs.items():
        model = model.to(dev)

        # Every arm takes (x_seq, edge_index) with x_seq shaped [N, W, F];
        # the no-graph arm simply ignores the edges. needs_graph therefore
        # only decides whether edge_index is populated or empty.
        # Feature width differs per arm (the lag model is wider), so read it
        # off the model rather than assuming F. Set N_FEATURES_BY_ARCH to
        # override if attribute discovery fails on your classes.
        n_feat = N_FEATURES_BY_ARCH.get(name)
        if n_feat is None:
            for probe in ("encoder", "spatial_layer", "mlp", "input_proj"):
                mod = getattr(model, probe, None)
                if mod is None:
                    continue
                for p in mod.parameters():
                    if p.dim() == 2:
                        n_feat = p.shape[1]
                        break
                if n_feat:
                    break
        if n_feat is None:
            print(f"  {name}: could not infer input width; "
                  f"add it to N_FEATURES_BY_ARCH")
            continue

        x = torch.randn(N, W, n_feat, device=dev)
        ei = ei_full if needs_graph else ei_empty

        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        lossf = torch.nn.MSELoss()
        args = (x, ei)

        def step():
            opt.zero_grad()
            out = model(*args)
            out = out[0] if isinstance(out, (tuple, list)) else out
            out = out.squeeze(-1) if out.dim() > 1 else out
            loss = lossf(out, target)
            loss.backward()
            opt.step()

        for _ in range(n_warmup):
            step()
        if dev.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(n_timed):
            step()
        if dev.type == "cuda":
            torch.cuda.synchronize()
        train_s = (time.perf_counter() - t0) / n_timed

        model.eval()
        with torch.no_grad():
            for _ in range(n_warmup):
                model(*args)
            if dev.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(n_timed):
                model(*args)
            if dev.type == "cuda":
                torch.cuda.synchronize()
            infer_ms = (time.perf_counter() - t0) / n_timed * 1000.0

        rows.append(dict(
            architecture=name, params=count_params(model),
            n_features=n_feat, uses_graph=needs_graph,
            train_s_per_step=round(train_s, 4),
            inference_ms=round(infer_ms, 2),
            device=str(dev), n_nodes=N, config=cfg,
        ))
        print(f"  {name:16s} params={rows[-1]['params']:>9,}  F={n_feat:<3} "
              f"train={train_s*1000:7.1f} ms/step  infer={infer_ms:6.2f} ms")
        model.to("cpu")
    return rows


# =============================================================================
# SECTION 4 — map figure export
# =============================================================================

def export_map_arrays(cfg, predictions=None):
    """Write per-node observed / predicted / residual for the map figure.

    predictions: (N, T) array from the chosen model, or None. If None the
    file still gets observed values and block ids, so you can plot the target
    field and the block layout while you sort out which model to map.

    Averaged over valid instants per node -- a mean night-LST field, which is
    what a static three-panel map should show. Residual is predicted minus
    observed, so warm colours mean the model runs hot.
    """
    d = load_config(cfg)
    y, valid, pos = d["y"], d["valid"], d["pos"]
    n_valid = valid.sum(axis=1)

    with np.errstate(invalid="ignore"):
        obs = np.where(n_valid > 0, np.nansum(np.where(valid, y, 0), 1) / np.maximum(n_valid, 1), np.nan)

    out = pd.DataFrame({
        "node_id": d["node_id"],
        "x": pos[:, 0], "y": pos[:, 1],
        "block": d["block"], "split": d["split"],
        "n_valid": n_valid, "observed": obs,
    })

    if predictions is not None:
        pv = np.where(valid, predictions, 0)
        pred = np.where(n_valid > 0, pv.sum(1) / np.maximum(n_valid, 1), np.nan)
        out["predicted"] = pred
        out["residual"] = pred - obs

    path = os.path.join(OUT_DIR, f"map_export_{cfg}.csv")
    out.to_csv(path, index=False)
    print(f"  wrote {path}  ({len(out)} nodes, "
          f"{int((n_valid > 0).sum())} with at least one retrieval)")
    return out


# =============================================================================
# SECTION 5 — driver
# =============================================================================

def check_adapter(cfg):
    """Validate load_config() before spending time on a full run."""
    d = load_config(cfg)
    N, T, W, F = d["X"].shape
    print(f"config      : {cfg}")
    print(f"X           : {d['X'].shape}  (N={N} nodes, T={T} instants, "
          f"W={W} window, F={F} features)")
    print(f"y           : {d['y'].shape}")
    print(f"valid       : {d['valid'].shape}  "
          f"{d['valid'].mean()*100:.1f}% genuinely retrieved")
    print(f"pos         : {d['pos'].shape}  extent "
          f"{np.ptp(d['pos'][:,0]):.0f} x {np.ptp(d['pos'][:,1]):.0f} m")
    print(f"blocks      : {len(np.unique(d['block']))} distinct")
    for s in ("train", "val", "test"):
        print(f"  {s:5s}: {(d['split']==s).sum():4d} nodes")
    assert F == len(FEATURE_NAMES), (
        f"F={F} but FEATURE_NAMES has {len(FEATURE_NAMES)} entries")
    assert d["valid"].dtype == bool, "valid must be boolean"
    ymin, ymax = np.nanmin(d["y"][d["valid"]]), np.nanmax(d["y"][d["valid"]])
    print(f"target range: {ymin:.1f} to {ymax:.1f} "
          f"({'looks like Kelvin' if ymin > 200 else 'looks like Celsius'})")
    print("adapter OK")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    configs = load_configs()

    print("\n== tree baselines ==")
    rows = []
    for cfg in configs:
        rows += run_tree_baselines(cfg)
    tb = pd.DataFrame(rows)
    tb.to_csv(os.path.join(OUT_DIR, "tree_baselines.csv"), index=False)

    print("\n-- mean over seeds --")
    print(tb.groupby(["config", "model"])[["rmse", "spatial_r", "r2"]]
            .agg(["mean", "std"]).round(3).to_string())

    print("\n== model cost ==")
    cost = measure_cost(configs[0])
    if cost:
        pd.DataFrame(cost).to_csv(
            os.path.join(OUT_DIR, "model_cost.csv"), index=False)

    print("\n== map export ==")
    export_map_arrays(MAP_CONFIG or configs[0])

    print(f"\ndone -> {OUT_DIR}")


if __name__ == "__main__":
    import sys
    if len(sys.argv) > 2 and sys.argv[1] == "--check":
        check_adapter(sys.argv[2])
    else:
        main()

## 11 · Structureless reference and error decomposition

Splits MSE exactly into level and spatial-anomaly components. The structureless reference — each night's observed citywide mean at every node — has zero level error, so its RMSE equals the pooled spatial standard deviation.

In [ ]:
# ============================================================
# Closes G1, G2, G3 from the draft.
#
#   G1  oracle RMSE, all six configs           -> Table I, Abstract
#   G2  spread ratio for the tree models       -> contribution 3
#   G3  mean / anomaly error decomposition     -> explains the 2.4x gap
#
# The decomposition is exact. For each night t, split the error into the
# field-mean part and the spatial-anomaly part:
#
#     MSE_total = MSE_level + MSE_anomaly
#
# where MSE_level is the node-count-weighted mean of (pred_mean_t -
# obs_mean_t)^2 and MSE_anomaly is the mean squared error of the
# within-night anomalies. The cross term vanishes because anomalies sum
# to zero within each night. Nothing is approximated.
#
# Two things fall straight out of this:
#   - the oracle predicts obs_mean_t exactly, so MSE_level = 0 and
#     MSE_anomaly = the spatial variance. Oracle RMSE IS the pooled
#     spatial sd, which is why it is the right structureless reference.
#   - if the tree/neural RMSE gap sits in MSE_level, the two models have
#     comparable spatial skill and differ only in level tracking. That
#     is the paper's thesis in one table.
#
# Run after the grsl_supplements adapter cell, so load_config,
# knn_indices, add_spatial_lag, build_design_matrix, stack_rows and
# candidate_models are defined.
# ============================================================
import glob
import os

import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/UHI/PhyRes_MODIS/grsl_supp"
SEED = 0


# ---------------------------------------------------------------
# core: per-night level / anomaly split
# ---------------------------------------------------------------
def decompose(y, pred, mask):
    """Exact split of MSE into level and anomaly components.

    y, pred, mask: (N, T). mask marks scored (node, night) pairs.
    Returns RMSEs in the units of y, plus the spread ratio.
    """
    n_t = mask.sum(axis=0)                     # nodes scored per night
    keep = n_t >= 5                       # matches the notebook's threshold
    if keep.sum() == 0:
        return dict(rmse=np.nan, rmse_level=np.nan, rmse_anom=np.nan,
                    spread_ratio=np.nan, sd_obs=np.nan, sd_pred=np.nan,
                    n_nights=0)

    ym, pm, so, sp, w = [], [], [], [], []
    sq_anom = 0.0
    for t in np.flatnonzero(keep):
        m = mask[:, t]
        a, b = y[m, t], pred[m, t]
        ym.append(a.mean()); pm.append(b.mean())
        so.append(a.std());  sp.append(b.std())
        w.append(m.sum())
        sq_anom += (((b - b.mean()) - (a - a.mean())) ** 2).sum()

    ym, pm, so, sp = map(np.asarray, (ym, pm, so, sp))
    w = np.asarray(w, dtype=float)
    tot = w.sum()

    mse_level = float((w * (pm - ym) ** 2).sum() / tot)
    mse_anom = float(sq_anom / tot)
    ok = so > 1e-9
    return dict(
        rmse=float(np.sqrt(mse_level + mse_anom)),
        rmse_level=float(np.sqrt(mse_level)),
        rmse_anom=float(np.sqrt(mse_anom)),
        spread_ratio = float(np.sqrt((w * sp**2).sum() / (w * so**2).sum())),
        sd_obs=float(np.mean(so[ok])) if ok.any() else np.nan,
        sd_pred=float(np.mean(sp[ok])) if ok.any() else np.nan,
        n_nights=int(keep.sum()),
    )


def oracle_scores(y, mask):
    """Structureless reference: predict each night's observed held-out
    mean at every node. Zero spatial anomaly by construction."""
    pred = np.zeros_like(y, dtype=np.float64)
    for t in range(y.shape[1]):
        m = mask[:, t]
        if m.sum() > 0:
            pred[:, t] = y[m, t].mean()
    return decompose(y, pred, mask), pred


# ---------------------------------------------------------------
# tree predictions on the test blocks
# ---------------------------------------------------------------
def tree_predictions(d, feat="+lag", family="RandomForest"):
    """Refit the val-selected model and return (N, T) test predictions."""
    X, y, valid, pos, split = (d["X"], d["y"], d["valid"], d["pos"],
                               d["split"])
    tr, va, te = split == "train", split == "val", split == "test"

    D = build_design_matrix(X)
    if feat == "+lag":
        D = add_spatial_lag(D, knn_indices(pos, N_LAG_NEIGHBOURS))

    Xtr, ytr, _ = stack_rows(D, y, valid, tr)
    Xva, yva, _ = stack_rows(D, y, valid, va)
    Xte, yte, idx_te = stack_rows(D, y, valid, te)

    best, best_rmse, label = None, np.inf, None
    for lab, cand in candidate_models(SEED)[family]:
        cand.fit(Xtr, ytr)
        r = float(np.sqrt(((cand.predict(Xva) - yva) ** 2).mean()))
        if r < best_rmse:
            best, best_rmse, label = cand, r, lab

    pred = np.zeros_like(y, dtype=np.float64)
    pred[idx_te[:, 0], idx_te[:, 1]] = best.predict(Xte)
    mask = np.zeros_like(valid)
    mask[idx_te[:, 0], idx_te[:, 1]] = True
    return pred, mask, label


# ---------------------------------------------------------------
# main
# ---------------------------------------------------------------
os.makedirs(OUT_DIR, exist_ok=True)
rows = []

for cfg in load_configs():
    print(f"\n### {cfg}")
    d = load_config(cfg)
    y, valid = d["y"], d["valid"]
    te_mask = (d["split"] == "test")[:, None] & valid

    orc, _ = oracle_scores(y, te_mask)
    rows.append(dict(config=cfg, model="oracle (level only)", **orc))
    print(f"  oracle           RMSE {orc['rmse']:.3f}  "
          f"(level {orc['rmse_level']:.3f} / anom {orc['rmse_anom']:.3f})  "
          f"obs sd {orc['sd_obs']:.3f}")

    for family in ("RandomForest", "HistGBR"):
        for feat in ("plain", "+lag"):
            pred, mask, lab = tree_predictions(d, feat, family)
            s = decompose(y, pred, mask)
            n_t = mask.sum(0)
            keep = np.flatnonzero(n_t >= 5)
            dt = np.array([pred[mask[:, t], t].mean() - y[mask[:, t], t].mean()
                        for t in keep])
            bias, spread = dt.mean(), dt.std()
            frac = bias**2 / (bias**2 + spread**2)
            print(f"    level error: bias {bias:+.3f} °C, night-to-night sd "
                f"{spread:.3f} °C, {100*frac:.0f}% systematic")
            rows.append(dict(config=cfg, model=f"{family} {feat}",
                             selected=lab, **s))
            print(f"  {family:<14}{feat:<6} RMSE {s['rmse']:.3f}  "
                  f"(level {s['rmse_level']:.3f} / anom {s['rmse_anom']:.3f})  "
                  f"spread {s['spread_ratio']:.3f}")

df = pd.DataFrame(rows)
df.to_csv(f"{OUT_DIR}/oracle_and_decomposition.csv", index=False)

# ---- Table I skeleton --------------------------------------------
print("\n" + "=" * 74)
print("TABLE I — oracle vs best tree")
print("=" * 74)
o = df[df.model == "oracle (level only)"].set_index("config")
b = (df[df.model.str.contains(r"\+lag")]
       .sort_values("rmse").groupby("config").first())
tab = pd.DataFrame({
    "spatial_sd": o["sd_obs"],
    "oracle_rmse": o["rmse"],
    "model_rmse": b["rmse"],
    "model_level": b["rmse_level"],
    "model_anom": b["rmse_anom"],
    "spread_ratio": b["spread_ratio"],
    "beats_oracle_by": o["rmse"] - b["rmse"],
}).round(3)
print(tab.to_string())

print("\nreading this:")
print("  oracle_rmse ~= spatial_sd            -> confirms the identity;")
print("                                          if they differ, the mask")
print("                                          is inconsistent somewhere.")
print("  beats_oracle_by <= 0                 -> that model shows NO spatial")
print("                                          skill over a flat field.")
print("  model_anom vs oracle_rmse            -> the real spatial skill:")
print("                                          anomaly error against the")
print("                                          spatial sd it had to explain.")
print("  spread_ratio < 1 everywhere          -> the variance deficit,")
print("                                          now measured for trees (G2).")
print(f"\nwrote {OUT_DIR}/oracle_and_decomposition.csv")

# ---- optional: same decomposition for a saved neural model --------
print("\n" + "-" * 74)
print("neural checkpoints found (for G3 — run the block below on one):")
for p in sorted(glob.glob(f"{OUT_ROOT}/*/*_Models/*.pt")):
    print("  ", p)
print("""
To decompose a neural model, rebuild it and collect (N, T) predictions
over the test loader, then call decompose(y, pred, mask) with the same
te_mask used above. The number that matters is rmse_level: if the
neural model's RMSE penalty against the trees sits there rather than in
rmse_anom, the two have comparable spatial skill and differ only in
tracking the nightly level -- which is the paper's argument, measured.
""")

## 12 · Neural decomposition and signed level error

Loads the saved sweep checkpoints; nothing is retrained. Produces Table II of the letter.

In [ ]:
# ============================================================
# Neural model decomposition + G7 signed level error.
#
# Loads the saved sweep checkpoints (nothing is retrained), collects
# (N, T) test predictions, and splits the error the same way as the
# trees so the two are directly comparable.
#
# G7: the level error is split further into a constant bias and
# night-to-night scatter. MSE_level = bias^2 + variance exactly, so
# frac_systematic says how much of the level penalty a single per-config
# intercept would remove.
#
#   frac near 0    -> unbiased, just noisy about the level.
#                     Say so in one sentence and move on.
#   frac above 0.5 -> a systematic warm/cold offset. Worth stating: a
#                     one-parameter correction would close most of the
#                     RMSE gap between the model classes while their
#                     spatial skill was equal throughout. That sharpens
#                     the paper's argument rather than weakening it.
#
# The test loader uses shuffle=False and batch_size=1, so batch j is
# window j and lines up with load_config's window axis (verified against
# make_loaders).
#
# Run after grsl_finish (needs decompose) and cells 2-9.
# ============================================================
import os

import numpy as np
import pandas as pd
import torch

ARM = "Full"            # Full | NoGraph | NoMorph | NoAtmos
OUT_CSV = f"{OUT_ROOT}/grsl_supp/neural_decomposition.csv"

ARM_DROP = {"Full": [], "NoGraph": [], "NoMorph": MORPH_IDX,
            "NoAtmos": ATMOS_IDX}


def neural_predictions(cfg_dict, D, arm=ARM):
    """(N, T) test predictions in °C, plus the mask of scored cells."""
    city, month = cfg_dict["city"], cfg_dict["month"]
    path = f"{OUT_ROOT}/{city}/{month}_Models/{month}_{arm}.pt"
    if not os.path.exists(path):
        return None, None, f"missing {os.path.basename(path)}"

    ck = torch.load(path, map_location=device)
    kind = "NoGraph" if arm == "NoGraph" else "PhyRes"
    raw = build_model(kind, len(FEATURE_COLS), HP)
    raw.load_state_dict(ck["state_dict"])
    raw = raw.to(device).eval()

    # the checkpoint carries its own feature mask; prefer it, since the
    # arm definition belongs with the weights
    fmask = ck.get("feature_mask")
    fmask = (fmask.to(device) if fmask is not None
             else make_feature_mask(ARM_DROP[arm]).to(device))

    L = D["pruned"]
    N = D["n_nodes"]
    T = len(L["test"].dataset)
    pred = np.zeros((N, T), dtype=np.float64)
    mask = np.zeros((N, T), dtype=bool)

    with torch.no_grad():
        for j, batch in enumerate(L["test"]):          # shuffle=False
            batch = batch.to(device)
            x = batch.x * fmask
            fp, _, _ = raw(x, batch.edge_index)
            pred[:, j] = (fp * D["temp_std"] + D["temp_mean"]).cpu().numpy()
            mask[:, j] = batch.loss_mask.cpu().numpy()
    return pred, mask, None


def level_error_split(y, pred, mask, min_nodes=5):
    """G7: signed level error per night -> bias, scatter, systematic share."""
    keep = np.flatnonzero(mask.sum(0) >= min_nodes)
    if keep.size == 0:
        return dict(level_bias=np.nan, level_sd=np.nan,
                    frac_systematic=np.nan)
    dt = np.array([pred[mask[:, t], t].mean() - y[mask[:, t], t].mean()
                   for t in keep])
    bias, spread = float(dt.mean()), float(dt.std())
    denom = bias ** 2 + spread ** 2
    return dict(level_bias=round(bias, 3), level_sd=round(spread, 3),
                frac_systematic=(round(bias ** 2 / denom, 3)
                                 if denom > 0 else np.nan))


rows = []
for cfg in CONFIGS:
    label = f"{cfg['city']}_{cfg['month']}"
    d = load_config(label)
    D = prepare(cfg, HP)

    pred, mask, err = neural_predictions(cfg, D)
    if err:
        print(f"  {label:<22} {err}")
        continue

    y = d["y"]
    if pred.shape != y.shape:
        print(f"  {label:<22} shape mismatch {pred.shape} vs {y.shape} "
              f"-- window alignment is off, do not use these numbers")
        continue

    s = decompose(y, pred, mask)
    g7 = level_error_split(y, pred, mask)
    rows.append(dict(config=label, model=f"neural {ARM}", **s, **g7))
    print(f"  {label:<22} RMSE {s['rmse']:.3f}  "
          f"(level {s['rmse_level']:.3f} / anom {s['rmse_anom']:.3f})  "
          f"spread {s['spread_ratio']:.3f}")
    print(f"  {'':<22} level error: bias {g7['level_bias']:+.3f} °C, "
          f"night-to-night sd {g7['level_sd']:.3f} °C, "
          f"{100*g7['frac_systematic']:.0f}% systematic")

nd = pd.DataFrame(rows)
nd.to_csv(OUT_CSV, index=False)

# ---- side by side with the trees ---------------------------------
tree = pd.read_csv(f"{OUT_ROOT}/grsl_supp/oracle_and_decomposition.csv")
orc = tree[tree.model == "oracle (level only)"].set_index("config")
best = (tree[tree.model.str.contains(r"\+lag")]
          .sort_values("rmse").groupby("config").first())
neu = nd.set_index("config")

print("\n" + "=" * 82)
print("LEVEL vs ANOMALY — where each model spends its error")
print("=" * 82)
cmp = pd.DataFrame({
    "oracle_rmse": orc["rmse"],
    "tree_rmse": best["rmse"],
    "tree_level": best["rmse_level"],
    "tree_anom": best["rmse_anom"],
    "neural_rmse": neu["rmse"],
    "neural_level": neu["rmse_level"],
    "neural_anom": neu["rmse_anom"],
    "neural_bias": neu["level_bias"],
    "pct_systematic": (100 * neu["frac_systematic"]).round(0),
}).round(3)
cmp["anom_gap"] = (cmp["neural_anom"] - cmp["tree_anom"]).round(3)
cmp["level_gap"] = (cmp["neural_level"] - cmp["tree_level"]).round(3)
print(cmp.to_string())

print("\nreading this:")
print("  level_gap >> anom_gap   -> comparable SPATIAL skill, different")
print("                             level tracking. RMSE ranks them on the")
print("                             wrong thing. This is the paper's claim.")
print("  pct_systematic high     -> a per-config intercept would remove most")
print("                             of the level penalty. Say so; it makes")
print("                             the argument stronger, not weaker.")
print("  pct_systematic low      -> level error is night-to-night scatter,")
print("                             not calibration. One sentence, move on.")
print(f"\nwrote {OUT_CSV}")

## 13 · Physics prior with and without the constraint

Tests whether the atmospheric/morphological decomposition survives at `alpha=0, gamma=0`.

In [ ]:
# ============================================================
# Does the decomposition survive without the prior forcing it?
#
# The question this exists to answer: on ERA5, rBldg looked like a
# finding and turned out to be the gamma dial -- it ran monotonically
# from -0.363 at gamma=0 to +0.697 at gamma=0.1. Nothing since the
# MODIS pivot has retested that. If the paper claims an interpretable
# atmospheric/morphological separation, this is the check a reviewer
# will ask for.
#
# Trimmed to ~20 min:
#   - with_prior arm is LOADED from the sweep checkpoints, not retrained
#   - only the gamma=0 arm actually trains
#   - Delhi only (both seasons); state the two-config scope in the paper
#
# Set CONFIG_FILTER = None and RETRAIN_WITH_PRIOR = True for the full
# six-config version if you ever want it.
#
# Run after cells 2, 3, 4, 5, 7, 8, 9.
# ============================================================
import os

import numpy as np
import pandas as pd
import torch

CONFIG_FILTER = "Delhi"        # city name, or None for all six
RETRAIN_WITH_PRIOR = False     # False -> load {month}_Full.pt from the sweep
OUT_CSV = f"{OUT_ROOT}/prior_cost.csv"

# must match cell 9, since the loaded checkpoints were trained at these
PRIOR_ALPHA, PRIOR_GAMMA = 1.0, 0.03


def decomposition_stats(model, loader, temp_std):
    """Spread of each head, and whether t_res tracks building cover.

    t_phys and t_res come back in normalised units, so both stds are
    scaled to degrees. r_res_bld uses raw_last[:, COVER_IDX] -- the same
    untransformed cover the push-pull term reads -- so it is directly
    comparable to rBldg in the main sweep.
    """
    model.eval()
    ph, rs, bl = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            m = batch.loss_mask
            if m.sum() < 5:
                continue
            _, t_phys, t_res = model(batch.x, batch.edge_index)
            ph.append(t_phys[m].cpu().numpy())
            rs.append(t_res[m].cpu().numpy())
            bl.append(batch.raw_last[m, COVER_IDX].cpu().numpy())
    if not ph:
        return dict(t_phys_std=np.nan, t_res_std=np.nan, r_res_bld=np.nan)

    ph = np.concatenate(ph)
    rs_ = np.concatenate(rs)
    bl = np.concatenate(bl)
    ok = np.isfinite(rs_) & np.isfinite(bl)
    degenerate = rs_[ok].std() < 1e-9 or bl[ok].std() < 1e-9
    return dict(
        t_phys_std=float(ph.std() * temp_std),
        t_res_std=float(rs_.std() * temp_std),
        r_res_bld=(np.nan if degenerate
                   else float(np.corrcoef(rs_[ok], bl[ok])[0, 1])),
    )


configs = [c for c in CONFIGS
           if CONFIG_FILTER is None or c["city"] == CONFIG_FILTER]

print("=" * 72)
print(f"PRIOR TEST — {len(configs)} config(s), "
      f"with_prior {'retrained' if RETRAIN_WITH_PRIOR else 'loaded'}")
print("=" * 72)

rows = []
for cfg in configs:
    city, month = cfg["city"], cfg["month"]
    ckpt_dir = f"{OUT_ROOT}/{city}/{month}_Models"
    print(f"\n### {city} — {month}")

    D = prepare(cfg, HP)
    L = D["pruned"]
    n_feat = len(FEATURE_COLS)
    fmask = make_feature_mask([])            # Full arm: nothing dropped

    # ---- arm 1: with prior -------------------------------------------
    ckpt_path = f"{ckpt_dir}/{month}_Full.pt"
    if not RETRAIN_WITH_PRIOR and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        raw = build_model("PhyRes", n_feat, HP)
        raw.load_state_dict(ck["state_dict"])
        # prep is the feature-mask closure; rebuild it rather than
        # trusting a pickled one
        _, prep = train_masked(build_model("PhyRes", n_feat, HP),
                               L["train"], L["val"], epochs=0,
                               lr=LR, alpha=PRIOR_ALPHA, gamma=PRIOR_GAMMA,
                               weight_decay=1e-4, clip=0.5, tag="prep_only",
                               temp_std=D["temp_std"], feat_mask=fmask)
        arms = [("with_prior", PRIOR_ALPHA, PRIOR_GAMMA, raw, prep)]
        print(f"    loaded {os.path.basename(ckpt_path)} "
              f"(alpha={ck.get('alpha')}, gamma={ck.get('gamma')})")
        if ck.get("alpha") != PRIOR_ALPHA or ck.get("gamma") != PRIOR_GAMMA:
            print("    !! checkpoint alpha/gamma differ from PRIOR_ALPHA/GAMMA "
                  "-- the comparison is not clean")
    else:
        if not RETRAIN_WITH_PRIOR:
            print(f"    {ckpt_path} missing; training it instead")
        set_seed(HP["seed"])
        raw, prep = train_masked(build_model("PhyRes", n_feat, HP),
                                 L["train"], L["val"], HP["epochs"],
                                 lr=LR, alpha=PRIOR_ALPHA, gamma=PRIOR_GAMMA,
                                 weight_decay=1e-4, clip=0.5, tag="with_prior",
                                 temp_std=D["temp_std"], feat_mask=fmask)
        arms = [("with_prior", PRIOR_ALPHA, PRIOR_GAMMA, raw, prep)]

    # ---- arm 2: no prior, plain Huber --------------------------------
    set_seed(HP["seed"])
    raw0, prep0 = train_masked(build_model("PhyRes", n_feat, HP),
                               L["train"], L["val"], HP["epochs"],
                               lr=LR, alpha=0.0, gamma=0.0,
                               weight_decay=1e-4, clip=0.5, tag="no_prior",
                               temp_std=D["temp_std"], feat_mask=fmask)
    torch.save({"state_dict": raw0.state_dict(), "alpha": 0.0, "gamma": 0.0,
                "city": city, "month": month, "arm": "prior_no_prior",
                "temp_mean": D["temp_mean"], "temp_std": D["temp_std"]},
               f"{ckpt_dir}/prior_no_prior.pt")
    arms.append(("no_prior", 0.0, 0.0, raw0, prep0))

    # ---- evaluate both -----------------------------------------------
    for tag, a, g, r, p in arms:
        m = MaskedModel(r, p).to(device)
        loss_breakdown(m, L["val"], a, g, D["temp_std"], f"{city} {month} {tag}")
        met, _, _, _ = diagnostic_eval(
            m, L["test"], D["temp_mean"], D["temp_std"],
            f"{OUT_ROOT}/{city}/{month}_prior_{tag}_Scatter.png",
            f"{city} {month} — {tag}")
        sk = spatial_skill(m, L["test"], D["temp_mean"], D["temp_std"], tag)
        dec = decomposition_stats(m, L["test"], D["temp_std"])
        rows.append(dict(city=city, month=month, arm=tag, alpha=a, gamma=g,
                         rmse=met["rmse"], r2=met["r2"],
                         spatial_r=sk["spatial_r_mean"],
                         spread_ratio=sk["spatial_spread_ratio"], **dec))
        print(f"    {tag:<11} RMSE {met['rmse']:.3f}°C | "
              f"spatial r {sk['spatial_r_mean']:+.3f} | "
              f"t_phys sd {dec['t_phys_std']:.3f}°C | "
              f"t_res sd {dec['t_res_std']:.3f}°C | "
              f"r(t_res, bld) {dec['r_res_bld']:+.3f}")

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print("\n" + "=" * 72)
print("no_prior minus with_prior")
print("=" * 72)
w = df[df.arm == "with_prior"].set_index(["city", "month"])
n = df[df.arm == "no_prior"].set_index(["city", "month"])
print(pd.DataFrame({
    "d_rmse": n["rmse"] - w["rmse"],           # negative -> prior costs accuracy
    "d_spatial_r": n["spatial_r"] - w["spatial_r"],
    "r_res_bld_with": w["r_res_bld"],
    "r_res_bld_without": n["r_res_bld"],
    "t_phys_sd_with": w["t_phys_std"],
    "t_phys_sd_without": n["t_phys_std"],
}).round(3).to_string())

print("\nreading this:")
print("  r_res_bld_without near 0   -> t_res was the gamma dial again; do")
print("                                NOT claim an interpretable split.")
print("  r_res_bld_without positive -> the decomposition is real.")
print("  t_phys_sd near 0           -> the physical head collapsed; nothing")
print("                                to interpret regardless of r.")
print("  d_rmse < 0                 -> the prior costs accuracy AT ITS")
print("                                CURRENT UNTUNED SETTING. That is the")
print("                                only claim the run supports.")
print(f"\nwrote {OUT_CSV}")